In [ ]:
import os, sys
EVAL_DIR = "/home/sagemaker-user/PS1_Eval"
os.makedirs(EVAL_DIR, exist_ok=True)
print(f"Eval dir: {EVAL_DIR}")

# PS1 — Model Evaluation & Issue Fix (No MLflow)

Standalone evaluation notebook — retrains PS1 models with targeted fixes and evaluates comprehensively.
No MLflow logging. Best models saved to S3 via boto3.

| Issue | Device | Fix Applied |
|---|---|---|
| Optuna val/test gap (AUC 0.84 → 0.63) | TVM | TimeSeriesSplit in Optuna instead of random split |
| Extreme imbalance, low recall (0.07–0.16) | GATE | Threshold tuning sweep + moderate SMOTE (5%) |
| 0% positive rate on will_fail_3d | VALIDATOR | Try will_fail_7d and will_fail_14d targets |
| Dead columns (>90% null) | All | Drop BUS_ID, avg_kpi_value, max_kpi_value |
| RF massive overfit (train F1 0.82 → test 0.13) | All | Dropped RF; focus on LGB (generalises best) |

In [ ]:
# CELL 1 — Install
%pip install -q lightgbm xgboost catboost optuna scikit-learn shap imbalanced-learn boto3 pandas numpy matplotlib seaborn joblib

In [ ]:
# CELL 2 — Imports
import warnings, json, pickle, io, datetime
warnings.filterwarnings("ignore")

import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, confusion_matrix,
    precision_recall_curve, roc_curve, brier_score_loss
)
from sklearn.calibration import calibration_curve
from imblearn.over_sampling import SMOTE
import lightgbm as lgb

print(f"LightGBM : {lgb.__version__}")
print(f"Optuna   : {optuna.__version__}")
print("All imports OK")

In [ ]:
# CELL 3 — Config
S3_BUCKET        = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
S3_GOLD_PREFIX   = "chicago/gold/device_ps1_daily"
S3_MODEL_PREFIX  = "chicago/models/ps1_eval"
REGION           = "us-east-1"

TARGET_COL  = "will_fail_3d"
DEVICE_COL  = "mars_device_category"   # actual column name in Gold parquet
DATE_COL    = "event_date"             # Gold writes transit_day → aliased below

# Dead columns confirmed >90% null in PS1 training run
DEAD_COLS = ["BUS_ID", "avg_kpi_value", "max_kpi_value"]

# Maintenance priority: recall target before optimising precision
RECALL_TARGET = 0.55

s3 = boto3.client("s3", region_name=REGION)
print(f"Target          : {TARGET_COL}")
print(f"Device col      : {DEVICE_COL}")
print(f"Date col        : {DATE_COL}")
print(f"Dead cols drop  : {DEAD_COLS}")
print(f"Recall target   : {RECALL_TARGET:.0%}")
print(f"Model save path : s3://{S3_BUCKET}/{S3_MODEL_PREFIX}/")
# -- Clean label: planned maintenance event type IDs (TVM and GATE) --
# Excludes scheduled maintenance events from will_fail_3d so the model learns
# unplanned failures only. Source: Michael PS1_GATE_SageMaker.ipynb Cell 23.
# Source: Michael's PS1_GATE_SageMaker.ipynb Cell 23 EXCLUDE dict
# Shared codes (105,106,108,129,135,140): software-reset / depot-maintenance
# TVM-specific: 184=cash-cassette, 407=software-update, 507/516/521-523=scheduled-PM,
#               1406=preventive-card-reader-swap
TVM_PLANNED_EVENT_TYPES  = [105, 106, 108, 129, 135, 140,
                             184, 407, 507, 516, 521, 522, 523, 1406]
GATE_PLANNED_EVENT_TYPES = [105, 106, 108, 129, 135, 140, 151]

# ── Calendar-based temporal splits (preferred over fraction splits) ───────────
# These dates match the actual Gold data history for the Chicago project.
# Set both to None to fall back to 70/15/15 fraction-based splits.
TRAIN_END_DATE = "2025-10-01"   # training ends on this date (exclusive)
VAL_END_DATE   = "2026-01-01"   # validation ends on this date (exclusive)
EMBARGO_DAYS   = 3              # remove last N days from train/val to prevent
                                # label straddling across split boundaries

# ── Champion selection ────────────────────────────────────────────────────────
CHAMPION_AP_TOL = 0.02    # within this AP of best, prefer lower overfit gap

# ── Quality gate (prevents saving a model worse than a random guess) ──────────
QUALITY_RECALL_FLOOR = {"TVM": 0.80, "GATE": 0.70}
QUALITY_MIN_AP_RATIO = 1.05    # test_AP must exceed 1.05 x base_rate

print(f"Train end      : {TRAIN_END_DATE}")
print(f"Val end        : {VAL_END_DATE}")
print(f"Embargo days   : {EMBARGO_DAYS}")
print(f"Champion AP tol: {CHAMPION_AP_TOL}")
print(f"Quality recall floors: {QUALITY_RECALL_FLOOR}")

In [ ]:
# CELL 4 — Load Gold data from S3
def read_parquet_s3(bucket, prefix):
    paginator = s3.get_paginator("list_objects_v2")
    keys = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            if obj["Key"].endswith(".parquet"):
                keys.append(obj["Key"])
    print(f"  Found {len(keys)} parquet files under s3://{bucket}/{prefix}")
    dfs = []
    for key in sorted(keys):
        resp = s3.get_object(Bucket=bucket, Key=key)
        dfs.append(pd.read_parquet(io.BytesIO(resp["Body"].read())))
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print("Loading Gold PS1 data from S3 ...")
df_raw = read_parquet_s3(S3_BUCKET, S3_GOLD_PREFIX)

# Gold SQL writes transit_day; alias to event_date (same as PS1 main notebook)
if "transit_day" in df_raw.columns and DATE_COL not in df_raw.columns:
    df_raw = df_raw.rename(columns={"transit_day": DATE_COL})
    print(f"  Renamed transit_day → {DATE_COL}")

df_raw[DATE_COL] = pd.to_datetime(df_raw[DATE_COL])
df_raw = df_raw.sort_values(DATE_COL).reset_index(drop=True)

print(f"\nShape          : {df_raw.shape}")
print(f"Columns (first 10): {list(df_raw.columns[:10])}")
print(f"Date range     : {df_raw[DATE_COL].min().date()} → {df_raw[DATE_COL].max().date()}")
print(f"Device counts  :\n{df_raw[DEVICE_COL].value_counts().to_string()}")
print(f"\n{TARGET_COL} rate : {df_raw[TARGET_COL].mean():.2%}")
if "will_fail_7d" in df_raw.columns:
    print(f"will_fail_7d rate  : {df_raw['will_fail_7d'].mean():.2%}")
if "will_fail_14d" in df_raw.columns:
    print(f"will_fail_14d rate : {df_raw['will_fail_14d'].mean():.2%}")

In [ ]:
# CELL 4b — Clean label: recompute will_fail_3d excluding planned maintenance (TVM + GATE)
# ─────────────────────────────────────────────────────────────────────────────
# Source: Michael's PS1_GATE_SageMaker.ipynb Cell 23 EXCLUDE dict.
#
# Planned maintenance events are operator-scheduled; the device is intentionally
# taken offline. Including them in will_fail_3d teaches the model to predict
# planned work, not actual unplanned failures — which inflates the positive rate
# and lowers precision in the field.
#
# Shared codes (105,106,108,129,135,140): software-reset / depot-maintenance
# TVM-specific : 184,407,507,516,521,522,523,1406 (cash-cassette, PM, card-reader-swap)
# GATE-specific: 151 (gate-controller scheduled restart)
#
# Implementation: vectorised per-device-category unplanned flag, then
# forward rolling 3-day window — same logic as Michael's recompute_label(),
# no separate chargeable_failures table required (uses event_type_id in Gold).

_USE_CLEAN_LABEL_TVM  = False
_USE_CLEAN_LABEL_GATE = False

if "event_type_id" in df_raw.columns:
    print("event_type_id found — recomputing will_fail_3d_clean for TVM and GATE ...")
    _before = {
        cat: df_raw[df_raw[DEVICE_COL] == cat][TARGET_COL].mean()
        for cat in ["TVM", "GATE"]
        if cat in df_raw[DEVICE_COL].values
    }

    # ── Per-device planned-event exclusion (vectorised) ───────────────────────
    # Default: every row is unplanned (1).
    # Overwrite to 0 where the event_type_id matches the device category's planned list.
    df_raw["_unplanned"] = 1

    _tvm_rows  = df_raw[DEVICE_COL] == "TVM"
    _gate_rows = df_raw[DEVICE_COL] == "GATE"

    df_raw.loc[_tvm_rows  & df_raw["event_type_id"].isin(set(TVM_PLANNED_EVENT_TYPES)),
               "_unplanned"] = 0
    df_raw.loc[_gate_rows & df_raw["event_type_id"].isin(set(GATE_PLANNED_EVENT_TYPES)),
               "_unplanned"] = 0

    tvm_planned_rows  = int((_tvm_rows  & (df_raw["_unplanned"] == 0)).sum())
    gate_planned_rows = int((_gate_rows & (df_raw["_unplanned"] == 0)).sum())
    print(f"  TVM  planned rows zeroed : {tvm_planned_rows:,}")
    print(f"  GATE planned rows zeroed : {gate_planned_rows:,}")

    # ── Forward 3-day window per device ───────────────────────────────────────
    # Shift(-1) skips today; rolling(3) looks at the next 3 days.
    # max() = 1 if any unplanned event occurs in (t, t+3].
    df_raw = df_raw.sort_values([DEVICE_COL, DATE_COL]).copy()
    df_raw["will_fail_3d_clean"] = (
        df_raw.groupby(DEVICE_COL)["_unplanned"]
        .transform(lambda x: x.shift(-1).rolling(3, min_periods=1).max().fillna(0))
        .astype(int)
    )
    df_raw.drop(columns=["_unplanned"], inplace=True)

    # ── Report: positive rate before and after ─────────────────────────────────
    print("\n[Clean label summary]  Positive rate: original vs clean")
    for cat in ["TVM", "GATE"]:
        if cat not in df_raw[DEVICE_COL].values:
            continue
        _mask   = df_raw[DEVICE_COL] == cat
        _orig   = _before.get(cat, float("nan"))
        _clean  = df_raw.loc[_mask, "will_fail_3d_clean"].mean()
        _n_excl = len(TVM_PLANNED_EVENT_TYPES if cat == "TVM" else GATE_PLANNED_EVENT_TYPES)
        print(f"  {cat:<8}: original={_orig:.3%}  clean={_clean:.3%}  "
              f"reduction={_orig - _clean:.3%}  "
              f"({_n_excl} event types excluded)")

    _USE_CLEAN_LABEL_TVM  = True
    _USE_CLEAN_LABEL_GATE = True
    print("\n[ACTIVE] TVM  will use will_fail_3d_clean as its training target.")
    print("[ACTIVE] GATE will use will_fail_3d_clean as its training target.")
    print("  Both TVM and GATE labels now reflect UNPLANNED failures only.")

else:
    print("[Clean label] event_type_id not in Gold — using pre-computed will_fail_3d.")
    print("  Add event_type_id to the Gold SQL query to enable clean labels.")
    print(f"  TVM  exclusion list ({len(TVM_PLANNED_EVENT_TYPES)} types): {TVM_PLANNED_EVENT_TYPES}")
    print(f"  GATE exclusion list ({len(GATE_PLANNED_EVENT_TYPES)} types): {GATE_PLANNED_EVENT_TYPES}")
    print("  Expected recall improvement: 8-15% for both TVM and GATE after exclusion.")

In [ ]:
# CELL 5 — Column analysis + feature selection + lag/rolling feature engineering

df = df_raw.copy()

# ── Step 1: Drop dead columns ─────────────────────────────────────────────────
dead_present = [c for c in DEAD_COLS if c in df.columns]
if dead_present:
    df = df.drop(columns=dead_present)
    print(f"Dropped dead columns: {dead_present}\n")

# ── Step 1b: Deduplicate columns (parquet join artefact) ─────────────────────
# Feature-store parquets can carry duplicate column names from multi-table joins.
# Keep the first occurrence and drop all later copies before any dtype inspection.
_dup_cols = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
if _dup_cols:
    print(f"WARNING: duplicate columns found in source data — keeping first copy:")
    for _dc in _dup_cols:
        _n = (df.columns == _dc).sum()
        print(f"  {_dc!r}  x{_n}")
    df = df.loc[:, ~df.columns.duplicated(keep="first")]
    print(f"  -> After dedup: {df.shape[1]} columns remain\n")

# ── Step 2: Exclusion sets ────────────────────────────────────────────────────
ID_COLS = [DATE_COL, DEVICE_COL, "DEVICE_ID", "DEVICE_KEY", "DEVICE_SERIAL_NUMBER", "split"]
LABEL_LEAK_COLS = [
    "failure_level", "max_failure_level",
    "is_chargeable", "chargeable_outage_count",
    "will_fail_3d", "will_fail_7d", "will_fail_14d",
    "failure_date", "label_source",
]
HIGH_CARD_IDS = ["FACILITY_ID", "OPERATOR_ID", "TRANSIT_ARRAY_ID"]
ALL_EXCLUDE = set(ID_COLS + LABEL_LEAK_COLS + HIGH_CARD_IDS)

# ── Step 3: Coerce object columns that contain numeric data ───────────────────
# Parquet sometimes stores numeric columns as object/string dtype (mixed types,
# nullable integers, etc.). Those columns would otherwise be dropped silently
# from FEATURE_COLS because they fail the categorical cardinality check.
# Rule: if >50% of non-null values parse as float -> treat as numeric.
_coerced_to_numeric = []
for _col in df.select_dtypes(include="object").columns:
    if _col in ALL_EXCLUDE:
        continue
    _tried = pd.to_numeric(df[_col], errors="coerce")
    if _tried.notna().mean() > 0.5:
        df[_col] = _tried
        _coerced_to_numeric.append(_col)

if _coerced_to_numeric:
    print(f"Coerced {len(_coerced_to_numeric)} object->numeric columns:")
    for _c in _coerced_to_numeric:
        print(f"  {_c}")
else:
    print("No object columns needed numeric coercion.")

# ── Step 4: Drop 100%-null columns (zero signal regardless of dtype) ──────────
_null_pct = df.isnull().mean()
_all_null  = _null_pct[_null_pct == 1.0].index.tolist()
_all_null  = [c for c in _all_null if c not in ALL_EXCLUDE]
if _all_null:
    df = df.drop(columns=_all_null)
    print(f"\nDropped 100%-null columns: {_all_null}")

# ── Step 5: Cardinality report + encode low-cardinality categoricals ──────────
# Threshold 250 (not 200) to capture FACILITY_NAME (229 unique) — facility-level
# failure patterns are a real signal (some facilities maintain devices better).
CARDINALITY_THRESHOLD = 250

cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
cat_cols_feat = [c for c in cat_cols if c not in ALL_EXCLUDE]
print(f"\n{'Column':<45} {'Unique':>7}  {'Null%':>6}  Decision")
print("-" * 75)
USEFUL_CATS = []
for col in cat_cols_feat:
    nuniq    = df[col].nunique()
    null_pct = df[col].isna().mean() * 100
    if nuniq <= CARDINALITY_THRESHOLD:
        decision = f"KEEP (<={CARDINALITY_THRESHOLD} unique)"
        USEFUL_CATS.append(col)
    else:
        decision = f"SKIP ({nuniq} unique)"
    print(f"  {col:<43} {nuniq:>7}  {null_pct:>5.1f}%  {decision}")

from sklearn.preprocessing import LabelEncoder
label_encoders = {}
for col in USEFUL_CATS:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].fillna("__missing__").astype(str))
    label_encoders[col] = le

# ── Step 6: Base numeric FEATURE_COLS ─────────────────────────────────────────
# IMPORTANT: exclude USEFUL_CATS from num_features.
# Label-encoding converts USEFUL_CATS to int64 — without this exclusion they
# appear in BOTH num_features and USEFUL_CATS, creating duplicate column names
# that crash sklearn/narwhals (DuplicateError).
_useful_cats_set = set(USEFUL_CATS)
num_features = [c for c in df.columns
                if c not in ALL_EXCLUDE
                and c not in _useful_cats_set
                and pd.api.types.is_numeric_dtype(df[c])]
FEATURE_COLS = num_features + USEFUL_CATS
print(f"\nBase features: {len(num_features)} numeric + {len(USEFUL_CATS)} categorical = {len(FEATURE_COLS)} total")
print(f"  [Previously coerced numeric] {_coerced_to_numeric}")

# ── Step 7: Sort by device + date (required before shift/rolling) ─────────────
df = df.sort_values([DEVICE_COL, DATE_COL]).reset_index(drop=True)

# ── Step 8: Lag / rolling features (temporal — no target leakage) ─────────────
# All transforms use shift(1) so today's value is never part of any window.
# Tree models (LGB/XGB/CB) handle NaN natively — no fillna needed.
_LAG_DAYS  = [1, 3, 7]
_ROLL_WINS = [7, 14]
# Limit to first 15 numeric signals to avoid feature explosion.
# Sort by absolute Pearson correlation with target so the 15 most predictive
# operational metrics get lagged — not the arbitrary first-15 numeric columns.
# This ensures outage_min_7d, availability_pct_7d etc. are prioritised over
# static device fields that happen to appear first in df.columns.
_lag_corr  = df[num_features].corrwith(df[TARGET_COL]).abs().sort_values(ascending=False)
_lag_base  = [c for c in _lag_corr.index if c in df.columns][:15]
print(f"Lag base (top-15 by |r|): {_lag_base}")
lag_cols_added = []

for _col in _lag_base:
    _g = df.groupby(DEVICE_COL)[_col]
    for _d in _LAG_DAYS:
        _nc = f"{_col}_lag{_d}"
        df[_nc] = _g.transform(lambda x, d=_d: x.shift(d))
        lag_cols_added.append(_nc)
    for _w in _ROLL_WINS:
        _nc_m = f"{_col}_roll{_w}_mean"
        _nc_s = f"{_col}_roll{_w}_std"
        df[_nc_m] = _g.transform(lambda x, w=_w: x.shift(1).rolling(w, min_periods=2).mean())
        df[_nc_s] = _g.transform(lambda x, w=_w: x.shift(1).rolling(w, min_periods=2).std())
        lag_cols_added.extend([_nc_m, _nc_s])
    _nc_d = f"{_col}_delta1"
    df[_nc_d] = df[_col] - _g.transform(lambda x: x.shift(1))
    lag_cols_added.append(_nc_d)

print(f"Lag/rolling features added : {len(lag_cols_added)}")

# ── Step 9: Failure history features ──────────────────────────────────────────
_hist_src = TARGET_COL if TARGET_COL in df.columns else None
hist_cols_added = []

if _hist_src:
    _g_hist = df.groupby(DEVICE_COL)[_hist_src]
    for _w in [7, 14, 30]:
        _nc = f"failure_count_{_w}d"
        df[_nc] = _g_hist.transform(lambda x, w=_w: x.shift(1).rolling(w, min_periods=1).sum())
        hist_cols_added.append(_nc)
    df["failure_rate_30d"] = _g_hist.transform(
        lambda x: x.shift(1).rolling(30, min_periods=1).mean())
    hist_cols_added.append("failure_rate_30d")

    df["_sfail"]       = _g_hist.transform(lambda x: x.shift(1).fillna(0))
    df["_fail_anchor"] = df[DATE_COL].where(df["_sfail"] == 1)
    df["_last_fail"]   = df.groupby(DEVICE_COL)["_fail_anchor"].transform(lambda x: x.ffill())
    df["days_since_last_failure"] = (df[DATE_COL] - df["_last_fail"]).dt.days.fillna(999).clip(0, 999)
    df.drop(columns=["_sfail", "_fail_anchor", "_last_fail"], inplace=True)
    hist_cols_added.append("days_since_last_failure")
    print(f"Failure history features added : {len(hist_cols_added)}")

FEATURE_COLS = FEATURE_COLS + lag_cols_added + hist_cols_added

# Safety net: eliminate any remaining duplicates (preserves insertion order)
_before = len(FEATURE_COLS)
FEATURE_COLS = list(dict.fromkeys(FEATURE_COLS))
if len(FEATURE_COLS) < _before:
    print(f"[Safety dedup] Removed {_before - len(FEATURE_COLS)} duplicate entries from FEATURE_COLS")

print(f"Total FEATURE_COLS             : {len(FEATURE_COLS)}")
print(f"  Base            : {len(num_features) + len(USEFUL_CATS)}")
print(f"  Lag/rolling     : {len(lag_cols_added)}")
print(f"  Failure history : {len(hist_cols_added)}")

# ── Step 11: IsolationForest anomaly score as an additional feature ───────────
# Fits on training rows only (chronological first 70%) then transforms all rows.
# Contamination=0.02 flags ~2% of device-days as anomalous — the anomaly score
# (continuous, not binary) is more useful than the flag for gradient boosting.
# This captures multivariate outlier patterns not visible in any single feature.
from sklearn.ensemble import IsolationForest as _IF

_n_total = len(df)
_tr_mask = df.index < int(_n_total * 0.70)   # chronological train window

_if_features = [c for c in FEATURE_COLS if pd.api.types.is_numeric_dtype(df[c])]
_X_if = df[_if_features].fillna(df[_if_features].median())

# Subsample training set for speed (IF is O(n log n))
_X_if_tr = _X_if[_tr_mask]
_n_if_sample = min(50_000, len(_X_if_tr))
_if_sample_idx = _X_if_tr.sample(_n_if_sample, random_state=42).index

_iso = _IF(n_estimators=200, contamination=0.02, random_state=42, n_jobs=-1)
_iso.fit(_X_if.loc[_if_sample_idx])

# Anomaly score: higher = more anomalous (raw score negated from decision_function)
df["if_anomaly_score"] = -_iso.score_samples(_X_if)
df["if_anomaly_flag"]  = (_iso.predict(_X_if) == -1).astype(int)

FEATURE_COLS.append("if_anomaly_score")
print(f"IsolationForest anomaly score added   (contamination=0.02, n_sample={_n_if_sample:,})")
print(f"  Anomaly score: mean={df['if_anomaly_score'].mean():.4f}  "
      f"p95={df['if_anomaly_score'].quantile(0.95):.4f}")
print(f"  Anomaly flag : {df['if_anomaly_flag'].mean():.2%} device-days flagged")
# ── Device-level rolling failure rate ────────────────────────────────────────
# Captures whether this device is a CHRONIC failure source vs a first-time event.
# Computed from the clean label when available; lagged 1 day to avoid target leakage.
_rate_src = "will_fail_3d_clean" if "will_fail_3d_clean" in df.columns else TARGET_COL
df = df.sort_values([DEVICE_COL, DATE_COL]).copy()
df["device_fail_rate_30d"] = (
    df.groupby(DEVICE_COL)[_rate_src]
    .transform(lambda x: x.shift(1).rolling(30, min_periods=5).mean().fillna(0))
)
FEATURE_COLS.append("device_fail_rate_30d")
print(f"device_fail_rate_30d: mean={df['device_fail_rate_30d'].mean():.4f}  "
      f"(source={_rate_src}, lag=1d, window=30d, min_periods=5)")

# ── Interaction features: anomaly × recent failures ───────────────────────────
# anomaly_score × failures_Xd amplifies the risk signal for devices that are
# BOTH statistically anomalous AND in an active degradation pattern.
# Not all data versions have failures_Xd columns — conditional on presence.
_fail_roll_cols = sorted([c for c in df.columns
                          if "fail" in c.lower() and any(s in c for s in ["7d", "14d", "30d"])])
_inter_added = []
for _fc in _fail_roll_cols[:3]:   # cap at 3 to avoid feature explosion
    _ic = f"anomaly_x_{_fc}"
    df[_ic] = df["if_anomaly_score"] * df[_fc]
    FEATURE_COLS.append(_ic)
    _inter_added.append(_ic)
if _inter_added:
    print(f"Interaction features ({len(_inter_added)}): {_inter_added}")
else:
    print("  [Skip] No failures_Xd columns found — interaction features not created.")
    print("         (Add failures_7d / failures_14d / failures_30d in feature engineering)")
print(f"Total FEATURE_COLS (incl. anomaly + device-rate + interactions): {len(FEATURE_COLS)}")

# -- Step 10 (moved after IsolationForest): Device splits ----------------
# if_anomaly_score is appended to df in Step 11 above.
# Copying AFTER ensures df_tvm and df_gate inherit the anomaly feature.
# VALIDATOR removed from this analysis.
df_tvm  = df[df[DEVICE_COL] == "TVM"].copy().reset_index(drop=True)
df_gate = df[df[DEVICE_COL] == "GATE"].copy().reset_index(drop=True)

print(f"\nDevice splits (post-IsolationForest):")
print(f"  TVM  rows : {len(df_tvm):,}  | {TARGET_COL} rate: {df_tvm[TARGET_COL].mean():.2%}")
print(f"  GATE rows : {len(df_gate):,}  | {TARGET_COL} rate: {df_gate[TARGET_COL].mean():.2%}")
print(f"  if_anomaly_score in df_tvm  : {'if_anomaly_score' in df_tvm.columns}")
print(f"  if_anomaly_score in df_gate : {'if_anomaly_score' in df_gate.columns}")

In [ ]:
# CELL 5b - Feature diagnostics: run on TVM AND GATE separately
# -----------------------------------------------------------------------------
# Why per-device:
#   TVM (5% positive) vs GATE (0.63%) have very different MI landscapes.
#   A feature zero-MI for GATE might be the strongest signal for TVM, and vice
#   versa. Collinearity structure also differs between device categories.
#
# Outputs:
#   FEATURE_COLS       - globally cleaned (variance + TVM collinearity)
#   FEATURE_COLS_TVM   - same as FEATURE_COLS
#   FEATURE_COLS_GATE  - FEATURE_COLS minus features that are pure noise for GATE

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
import warnings

# ── Devices to diagnose ───────────────────────────────────────────────────────
_DIAG_PAIRS = [("TVM",  df_tvm,  TARGET_COL),
               ("GATE", df_gate, TARGET_COL)]

_mi_per_dev    = {}   # device -> pd.Series of MI scores
_zero_mi_dev   = {}   # device -> list of zero-MI features
_corr_drop_dev = {}   # device -> set of collinear features to drop

print("=" * 80)
print("FEATURE DIAGNOSTICS -- TVM and GATE (separate analysis)")
print("=" * 80)

for _dev, _df_dev, _tgt in _DIAG_PAIRS:
    print(f"\n{'─'*70}")
    print(f"Device: {_dev}  | rows={len(_df_dev):,}  | pos_rate={_df_dev[_tgt].mean():.3%}")
    print(f"{'─'*70}")

    _fc_avail = [c for c in FEATURE_COLS if c in _df_dev.columns]
    _diag     = _df_dev[_fc_avail + [_tgt]].copy()
    X_d       = _diag[_fc_avail]
    y_d       = _diag[_tgt]

    # 1. Missing values
    _null = (X_d.isnull().mean() * 100).sort_values(ascending=False)
    _high = _null[_null > 30]
    print(f"\n[Missing >30%]  {len(_high)} features")
    if len(_high):
        print(_high.head(10).to_string())

    # 2. Near-zero variance
    _varz    = X_d.var()
    _low_var = _varz[_varz < 1e-6].index.tolist()
    print(f"[Low variance]  {len(_low_var)} features (var < 1e-6)")

    # 3. Pearson r with target
    _tcorr = X_d.corrwith(y_d).abs().sort_values(ascending=False)
    print(f"\n[Pearson |r| vs target] Top 10:")
    print(_tcorr.head(10).to_string())
    print(f"  Features |r| < 0.01: {(_tcorr < 0.01).sum()}")

    # 4. Mutual Information
    print(f"\n[Mutual Information] Computing (may take ~30s for {_dev}) ...")
    _X_mi = X_d.fillna(X_d.median())
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _mi_scores = mutual_info_classif(_X_mi, y_d, random_state=42, n_jobs=-1)
    _mi_s = pd.Series(_mi_scores, index=_fc_avail).sort_values(ascending=False)
    _mi_per_dev[_dev]  = _mi_s
    _zero_mi           = _mi_s[_mi_s < 1e-6].index.tolist()
    _zero_mi_dev[_dev] = _zero_mi
    print(f"  Top 15 by MI:")
    print(_mi_s.head(15).to_string())
    print(f"  Zero-MI features: {len(_zero_mi)}")

    # 5. Multicollinearity (Pearson |r| > 0.95 between features)
    print(f"\n[Multicollinearity] |r| > 0.95 pairs ...")
    _corr_m = X_d.corr().abs()
    _upper  = _corr_m.where(np.triu(np.ones(_corr_m.shape), k=1).astype(bool))
    _hc     = [(c, r, _upper.loc[r, c])
                for c in _upper.columns
                for r in _upper.index
                if pd.notna(_upper.loc[r, c]) and _upper.loc[r, c] > 0.95]
    _hc.sort(key=lambda x: -x[2])
    print(f"  {len(_hc)} pairs found  (showing top 10):")
    for a, b, rv in _hc[:10]:
        print(f"    {a:<40}  {b:<40}  r={rv:.4f}")

    # Drop the lower-MI feature from each correlated pair
    _drop_c = set()
    for a, b, _ in _hc:
        if a not in _drop_c and b not in _drop_c:
            _drop = b if _mi_s.get(a, 0) >= _mi_s.get(b, 0) else a
            _drop_c.add(_drop)
    _corr_drop_dev[_dev] = _drop_c
    print(f"  Drop {len(_drop_c)} redundant features (lower MI in each pair)")

    # 6. Outlier summary (>5 sigma)
    _z     = ((X_d - X_d.mean()) / X_d.std()).abs()
    _opct  = (_z > 5).mean() * 100
    _ocols = _opct[_opct > 1].sort_values(ascending=False)
    print(f"\n[Outliers >5-sigma, >1% rows]  {len(_ocols)} features")
    if len(_ocols):
        print(_ocols.head(8).to_string())
        print("  Tree models handle outliers via splits.")
        print("  IQR clipping (med +/- 3*IQR) recommended for stacking meta-LR.")

print(f"\n{'='*80}")

# ── MI comparison table: TVM vs GATE ─────────────────────────────────────────
if "TVM" in _mi_per_dev and "GATE" in _mi_per_dev:
    _cmp = pd.DataFrame({
        "TVM_MI" : _mi_per_dev["TVM"],
        "GATE_MI": _mi_per_dev["GATE"],
    }).fillna(0)
    _cmp["delta_TVM_minus_GATE"] = _cmp["TVM_MI"] - _cmp["GATE_MI"]
    print("\n[MI COMPARISON] TVM vs GATE -- top 20 by GATE MI:")
    print(_cmp.sort_values("GATE_MI", ascending=False).head(20).to_string())

    # Features useful for TVM but zero-MI for GATE (noise to remove from GATE)
    _gate_only_zero = [c for c in _zero_mi_dev.get("GATE", [])
                        if c not in _zero_mi_dev.get("TVM", [])]
    print(f"\nFeatures useful for TVM but zero-MI for GATE: {len(_gate_only_zero)}")
    if _gate_only_zero:
        print("  Excluding from GATE training only:", _gate_only_zero[:10])
    _GATE_EXTRA_DROPS = set(_gate_only_zero) | _corr_drop_dev.get("GATE", set())
else:
    _GATE_EXTRA_DROPS = set()

# ── Global FEATURE_COLS: remove features that are noise for TVM ──────────────
# (near-zero variance + TVM-specific collinear drops)
_tvm_remove = set(_low_var) | _corr_drop_dev.get("TVM", set())
_BEFORE = len(FEATURE_COLS)
FEATURE_COLS = [c for c in FEATURE_COLS if c not in _tvm_remove]
print(f"\nGlobal FEATURE_COLS: {_BEFORE} -> {len(FEATURE_COLS)} "
      f"({_BEFORE - len(FEATURE_COLS)} removed: low-var + TVM collinear)")

# ── Per-device feature lists ──────────────────────────────────────────────────
FEATURE_COLS_TVM  = FEATURE_COLS   # already cleaned on TVM
FEATURE_COLS_GATE = [c for c in FEATURE_COLS if c not in _GATE_EXTRA_DROPS]

print(f"\n{'='*80}")
print(f"PER-DEVICE FEATURE LISTS")
print(f"  FEATURE_COLS_TVM  : {len(FEATURE_COLS_TVM)}")
print(f"  FEATURE_COLS_GATE : {len(FEATURE_COLS_GATE)}  "
      f"({len(FEATURE_COLS_TVM) - len(FEATURE_COLS_GATE)} GATE-specific drops)")
print(f"  [Drops for GATE]  : {sorted(_GATE_EXTRA_DROPS)[:10]}{'...' if len(_GATE_EXTRA_DROPS)>10 else ''}")
print(f"{'='*80}")

# ── Correlation heatmap (TVM -- top 20 by MI) ─────────────────────────────────
if "TVM" in _mi_per_dev:
    _top20 = _mi_per_dev["TVM"].head(20).index.tolist()
    _avail = [c for c in _top20 if c in df_tvm.columns]
    _hm    = df_tvm[_avail + [TARGET_COL]].corr()
    plt.figure(figsize=(14, 12))
    sns.heatmap(_hm, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
                linewidths=0.4, square=False)
    plt.title("TVM -- Correlation matrix (top 20 features by MI)", fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{EVAL_DIR}/tvm_correlation_heatmap.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Heatmap saved -> {EVAL_DIR}/tvm_correlation_heatmap.png")

In [ ]:
# CELL 6 - Evaluation helpers

def time_split(df, val_frac=0.15, test_frac=0.15,
               train_end=None, val_end=None, embargo_days=0,
               date_col=None):
    """Chronological train / val / test split.

    Calendar mode (train_end + val_end both set):
      train : date < train_end  minus embargo_days
      val   : train_end <= date < val_end minus embargo_days
      test  : date >= val_end
    Falls back to 70/15/15 fraction split if any partition is empty.

    Fraction mode (train_end=None):
      70 / 15 / 15 % by row count.
    """
    _dc = date_col or DATE_COL
    if train_end is not None and val_end is not None:
        _te = pd.Timestamp(train_end) - pd.Timedelta(days=embargo_days)
        _ve = pd.Timestamp(val_end)   - pd.Timedelta(days=embargo_days)
        tr = df[df[_dc] <  _te].copy()
        va = df[(df[_dc] >= pd.Timestamp(train_end)) & (df[_dc] < _ve)].copy()
        te = df[df[_dc] >= pd.Timestamp(val_end)].copy()
        if len(tr) > 0 and len(va) > 0 and len(te) > 0:
            return tr, va, te
        print(f"[time_split] Calendar split has empty partition — "
              f"train={len(tr)} val={len(va)} test={len(te)}. Falling back to fraction split.")
    n = len(df)
    v  = int(n * (1 - test_frac - val_frac))
    t  = int(n * (1 - test_frac))
    return df.iloc[:v].copy(), df.iloc[v:t].copy(), df.iloc[t:].copy()


def evaluate(name, y_true, y_prob, threshold=0.5):
    """Full metric suite at a given threshold — includes Accuracy and Brier."""
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    total = tn + fp + fn + tp
    return {
        "split"    : name,
        "threshold": round(threshold, 4),
        "Accuracy" : round((tp + tn) / total, 4),
        "AUC"      : round(roc_auc_score(y_true, y_prob), 4),
        "AP"       : round(average_precision_score(y_true, y_prob), 4),
        "Precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall"   : round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1"       : round(f1_score(y_true, y_pred, zero_division=0), 4),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Brier"    : round(brier_score_loss(y_true, y_prob), 4),
    }


def evaluate_splits(model, splits, threshold=0.5):
    """Evaluate a fitted model on train / val / test splits.

    splits: dict with keys 'Train'/'Val'/'Test', values: (X, y) tuples.
    """
    rows = []
    for split_name, (X, y) in splits.items():
        prob = model.predict_proba(X)[:, 1]
        rows.append(evaluate(split_name, y, prob, threshold=threshold))
    df_out = pd.DataFrame(rows)
    cols = ["split", "threshold", "Accuracy", "AUC", "AP",
            "Precision", "Recall", "F1", "TP", "FP", "FN", "TN", "Brier"]
    return df_out[cols]


def find_threshold(y_true, y_prob, recall_target=0.55):
    """Lowest threshold achieving recall_target."""
    precs, recs, threshs = precision_recall_curve(y_true, y_prob)
    for t, r, p in sorted(zip(threshs, recs[:-1], precs[:-1])):
        if r >= recall_target:
            return round(float(t), 4), round(float(p), 4), round(float(r), 4)
    return 0.10, None, None


def plot_eval(title, y_true, y_prob, threshold, save_path=None):
    """4-panel: ROC / PR / Calibration / Confusion Matrix."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    axes[0, 0].plot(fpr, tpr, "steelblue", lw=2, label=f"AUC={auc:.4f}")
    axes[0, 0].plot([0, 1], [0, 1], "k--", alpha=0.4)
    axes[0, 0].set(title="ROC Curve", xlabel="FPR", ylabel="TPR")
    axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.25)

    precs, recs, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    axes[0, 1].plot(recs, precs, "seagreen", lw=2, label=f"AP={ap:.4f}")
    axes[0, 1].axhline(y_true.mean(), color="k", ls="--", alpha=0.4, label="Baseline")
    axes[0, 1].set(title="Precision-Recall", xlabel="Recall", ylabel="Precision")
    axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.25)

    try:
        prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=15)
        axes[1, 0].plot(prob_pred, prob_true, "o-", color="darkorange", lw=2, label="Model")
        axes[1, 0].plot([0, 1], [0, 1], "k--", alpha=0.4, label="Perfect")
    except Exception:
        axes[1, 0].text(0.5, 0.5, "Calibration N/A", ha="center", va="center")
    axes[1, 0].set(title="Calibration", xlabel="Mean Predicted Prob",
                   ylabel="Fraction Positive")
    axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.25)

    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", ax=axes[1, 1], cmap="Blues",
                xticklabels=["Pred 0", "Pred 1"],
                yticklabels=["True 0", "True 1"])
    axes[1, 1].set_title(f"Confusion Matrix  (threshold={threshold})")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    plt.show()


def threshold_sweep(y_true, y_prob, steps=None):
    """Precision / recall / accuracy / F1 / TP / FP across thresholds."""
    if steps is None:
        steps = np.arange(0.05, 0.65, 0.05)
    rows = []
    for t in steps:
        m = evaluate(f"{t:.2f}", y_true, y_prob, threshold=t)
        rows.append({"threshold": t, "Accuracy": m["Accuracy"],
                     "Recall": m["Recall"], "Precision": m["Precision"],
                     "F1": m["F1"], "TP": m["TP"], "FP": m["FP"],
                     "FN": m["FN"]})
    return pd.DataFrame(rows)


# ── Ranking metrics (operations-relevant — maintenance teams work from lists) ──

def ranking_metrics(y_true, y_prob, ks=(0.01, 0.05, 0.10)):
    """precision@K, recall@K, lift@K — how well the model ranks failures first.

    ks: fractions of total population to inspect (e.g. 0.05 = top 5%).
    Lift@K = precision@K / base_rate. Target: lift@5% > 2.0 for useful dispatch.
    """
    y_arr = np.array(y_true)
    n = len(y_arr)
    base_rate = y_arr.mean()
    sorted_idx = np.argsort(y_prob)[::-1]
    y_sorted = y_arr[sorted_idx]
    total_pos = y_arr.sum()
    rows = {}
    for k in ks:
        top_n = max(1, int(np.ceil(n * k)))
        tp_k  = y_sorted[:top_n].sum()
        label = f"@{int(k * 100)}pct"
        rows[f"prec{label}"] = round(tp_k / top_n, 4)
        rows[f"rec{label}"]  = round(tp_k / max(1, total_pos), 4)
        rows[f"lift{label}"] = round((tp_k / top_n) / max(base_rate, 1e-9), 4)
    return rows


def print_ranking_table(device_name, y_true, y_prob):
    """Print precision/recall/lift at 1%, 5%, 10% for ops teams."""
    rm = ranking_metrics(y_true, y_prob)
    base = np.array(y_true).mean()
    print(f"\n{device_name} — Ranking metrics (base rate = {base:.3%})")
    print(f"  {'K':>6}  {'Prec@K':>8}  {'Rec@K':>8}  {'Lift@K':>8}")
    print(f"  {'─'*6}  {'─'*8}  {'─'*8}  {'─'*8}")
    for k in (0.01, 0.05, 0.10):
        lbl = f"@{int(k * 100)}pct"
        print(f"  {f'Top {int(k*100)}%':>6}  "
              f"{rm[f'prec{lbl}']:>8.4f}  "
              f"{rm[f'rec{lbl}']:>8.4f}  "
              f"{rm[f'lift{lbl}']:>8.2f}x")
    print("  [Ops target] Top 5% lift > 2.0 = model useful for ranked dispatch.")
    return rm


# ── Overfit gap + champion selection ─────────────────────────────────────────

def overfit_gap_suite(name, model, Xtr, ytr, Xva, yva, Xte, yte, threshold=0.5):
    """Train/val/test metrics + overfit gaps in one call."""
    def _p(X):
        return model.predict_proba(X)[:, 1]
    tr_ap  = average_precision_score(ytr, _p(Xtr))
    va_ap  = average_precision_score(yva, _p(Xva))
    te_ap  = average_precision_score(yte, _p(Xte))
    tr_auc = roc_auc_score(ytr, _p(Xtr))
    te_auc = roc_auc_score(yte, _p(Xte))
    row = evaluate(name, yte, _p(Xte), threshold=threshold)
    row["val_ap"]          = round(va_ap, 4)
    row["train_ap"]        = round(tr_ap, 4)
    row["overfit_gap_ap"]  = round(tr_ap - te_ap, 4)
    row["overfit_gap_auc"] = round(tr_auc - te_auc, 4)
    return row


def select_champion(model_results, ap_tol=None):
    """Pick champion from list of overfit_gap_suite() dicts.

    Strategy: highest val_ap; ties within ap_tol broken by lowest overfit_gap_ap.
    """
    _tol = ap_tol if ap_tol is not None else CHAMPION_AP_TOL
    if not model_results:
        return None
    best_val = max(r["val_ap"] for r in model_results)
    cands    = [r for r in model_results if r["val_ap"] >= best_val - _tol]
    return min(cands, key=lambda r: r.get("overfit_gap_ap", 999))


def quality_gate(device, model, Xte, yte, threshold,
                 recall_floor=None, min_ap_ratio=None):
    """Returns True (pass) if recall and AP both meet the minimum bars.

    Blocks S3 save when the model is no better than the base rate or too low recall.
    """
    _rf  = recall_floor  if recall_floor  is not None else QUALITY_RECALL_FLOOR.get(device, 0.55)
    _apr = min_ap_ratio  if min_ap_ratio  is not None else QUALITY_MIN_AP_RATIO
    proba = model.predict_proba(Xte)[:, 1]
    pred  = (proba >= threshold).astype(int)
    rec   = recall_score(yte, pred, zero_division=0)
    ap    = average_precision_score(yte, proba)
    base  = np.array(yte).mean()
    rec_ok = rec >= _rf
    ap_ok  = ap  >= base * _apr
    status = "PASS" if (rec_ok and ap_ok) else "FAIL"
    print(f"\n[Quality Gate] {device}  ->  {status}")
    print(f"  Recall : {rec:.4f}  (floor={_rf:.2f})  {'OK' if rec_ok else 'FAIL'}")
    print(f"  AP     : {ap:.4f}  (min={base * _apr:.4f} = {_apr}x base rate)  "
          f"{'OK' if ap_ok else 'FAIL'}")
    return rec_ok and ap_ok


print("Helper functions loaded:")
print("  time_split (calendar + fraction), evaluate, evaluate_splits,")
print("  find_threshold, plot_eval, threshold_sweep")
print("  ranking_metrics, print_ranking_table")
print("  overfit_gap_suite, select_champion, quality_gate")

---
## TVM — Evaluation & Fix
**Issue:** Optuna used random val split → val AUC 0.84 but test AUC 0.63 (leakage).  
**Fix:** Replace with `TimeSeriesSplit(n_splits=5)` so Optuna tunes on temporal folds.

In [ ]:
# CELL 7 — TVM data split (calendar-based if dates available, else 70/15/15)
# Uses will_fail_3d_clean as target when Cell 4b succeeded (_USE_CLEAN_LABEL_TVM=True).
_tvm_target = "will_fail_3d_clean" if _USE_CLEAN_LABEL_TVM else TARGET_COL
print(f"TVM target column   : {_tvm_target}")
print(f"TVM feature count   : {len(FEATURE_COLS_TVM)}")

train_tvm, val_tvm, test_tvm = time_split(
    df_tvm,
    train_end=TRAIN_END_DATE, val_end=VAL_END_DATE, embargo_days=EMBARGO_DAYS,
)

X_tr, y_tr = train_tvm[FEATURE_COLS_TVM], train_tvm[_tvm_target]
X_va, y_va = val_tvm[FEATURE_COLS_TVM],   val_tvm[_tvm_target]
X_te, y_te = test_tvm[FEATURE_COLS_TVM],  test_tvm[_tvm_target]

tvm_pos_rate         = y_tr.mean()
tvm_scale_pos_weight = (1 - tvm_pos_rate) / tvm_pos_rate

print(f"\nTVM Train : {len(X_tr):,}  (pos={y_tr.sum():,}  {tvm_pos_rate:.2%})")
print(f"TVM Val   : {len(X_va):,}  (pos={y_va.sum():,}  {y_va.mean():.2%})")
print(f"TVM Test  : {len(X_te):,}  (pos={y_te.sum():,}  {y_te.mean():.2%})")
print(f"scale_pos_weight : {tvm_scale_pos_weight:.2f}")

if _USE_CLEAN_LABEL_TVM:
    _orig_rate = df_tvm[TARGET_COL].mean()
    print(f"\n[Clean label] pos rate dropped {_orig_rate:.3%} -> {tvm_pos_rate:.3%}  "
          f"({_orig_rate - tvm_pos_rate:.3%} planned events removed)")

print(f"\nDate range - Train : {train_tvm[DATE_COL].min().date()} -> "
      f"{train_tvm[DATE_COL].max().date()}")
print(f"Date range - Val   : {val_tvm[DATE_COL].min().date()} -> "
      f"{val_tvm[DATE_COL].max().date()}")
print(f"Date range - Test  : {test_tvm[DATE_COL].min().date()} -> "
      f"{test_tvm[DATE_COL].max().date()}")

In [ ]:
# CELL 8 — TVM baseline LGB (no Optuna) — reference point
lgb_base_tvm = lgb.LGBMClassifier(
    objective="binary", n_estimators=1000, learning_rate=0.05,
    num_leaves=63, max_depth=-1, min_child_samples=50,
    colsample_bytree=0.8, subsample=0.8, subsample_freq=1,
    scale_pos_weight=tvm_scale_pos_weight, random_state=42, n_jobs=-1, verbose=-1
)
lgb_base_tvm.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=200)]
)

print("\nTVM Baseline LGB — Train / Val / Test (threshold=0.50):")
tvm_base_splits = {"Train": (X_tr, y_tr), "Val": (X_va, y_va), "Test": (X_te, y_te)}
df_tvm_base_splits = evaluate_splits(lgb_base_tvm, tvm_base_splits, threshold=0.50)
print(df_tvm_base_splits.to_string(index=False))
print("\n[Interpret] Large Train→Val/Test AUC gap = overfit. High Accuracy with low Recall = imbalance masking effect.")

# keep test-only row for later comparison table
m_base = evaluate("TVM-LGB-Baseline", y_te, lgb_base_tvm.predict_proba(X_te)[:, 1])

In [ ]:
# CELL 9 - TVM FIX: Optuna (TimeSeriesSplit, MedianPruner, metric=AP)
# Speed settings:
#   n_trials=20 + MedianPruner(startup=5) -> ~12-15 full trials typically
#   n_splits=3  (was 5) -> 3x fewer model fits inside each trial
#   timeout=600 -> hard 10-minute cap
#   early_stopping_rounds=20 (was 30) -> stop sooner on non-improving trials
from optuna.pruners import MedianPruner

X_train_full = pd.concat([X_tr, X_va]).reset_index(drop=True)
y_train_full = pd.concat([y_tr, y_va]).reset_index(drop=True)

tscv     = TimeSeriesSplit(n_splits=5)    # full 5-fold for stacking (Cell 9c)
tscv_opt = TimeSeriesSplit(n_splits=3)    # fast 3-fold for Optuna only

def tvm_objective(trial):
    params = {
        "objective"        : "binary",
        "n_estimators"     : trial.suggest_int("n_estimators",  300,  800),
        "learning_rate"    : trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves"       : trial.suggest_int("num_leaves",      20,  150),
        "max_depth"        : trial.suggest_int("max_depth",        3,   10),
        "min_child_samples": trial.suggest_int("min_child_samples",20, 200),
        "colsample_bytree" : trial.suggest_float("colsample_bytree",0.5, 1.0),
        "subsample"        : trial.suggest_float("subsample",      0.5, 1.0),
        "reg_alpha"        : trial.suggest_float("reg_alpha",    1e-4, 10.0, log=True),
        "reg_lambda"       : trial.suggest_float("reg_lambda",   1e-4, 10.0, log=True),
        "min_split_gain"   : trial.suggest_float("min_split_gain", 0.0,  1.0),
        "scale_pos_weight" : tvm_scale_pos_weight,
        "random_state": 42, "n_jobs": -1, "verbose": -1,
    }
    aps = []
    for step, (tr_idx, va_idx) in enumerate(tscv_opt.split(X_train_full)):
        Xf_tr = X_train_full.iloc[tr_idx]; yf_tr = y_train_full.iloc[tr_idx]
        Xf_va = X_train_full.iloc[va_idx]; yf_va = y_train_full.iloc[va_idx]
        m = lgb.LGBMClassifier(**params)
        m.fit(Xf_tr, yf_tr, eval_set=[(Xf_va, yf_va)],
              callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(0)])
        _ap = average_precision_score(yf_va, m.predict_proba(Xf_va)[:, 1])
        aps.append(_ap)
        trial.report(float(np.mean(aps)), step)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    return float(np.mean(aps))

print("Running Optuna TVM -- 3-fold x 20 trials, MedianPruner, metric=AP, timeout=600s ...")
study_tvm = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=1),
)
study_tvm.optimize(tvm_objective, n_trials=20, timeout=600, show_progress_bar=True)

best_p = study_tvm.best_params.copy()
best_p.update({"objective": "binary", "scale_pos_weight": tvm_scale_pos_weight,
               "random_state": 42, "n_jobs": -1, "verbose": -1})
model_tvm = lgb.LGBMClassifier(**best_p)
model_tvm.fit(X_train_full, y_train_full)
proba_tvm = model_tvm.predict_proba(X_te)[:, 1]

_pruned = len([t for t in study_tvm.trials
               if t.state == optuna.trial.TrialState.PRUNED])
print(f"\nBest CV AP  : {study_tvm.best_value:.4f}  "
      f"({_pruned}/{len(study_tvm.trials)} trials pruned early)")
print(f"Best params : depth={best_p.get('max_depth')}  "
      f"leaves={best_p.get('num_leaves')}  "
      f"reg_a={best_p.get('reg_alpha',0):.3f}  "
      f"reg_l={best_p.get('reg_lambda',0):.3f}")
print("\nTVM Optuna -- Train / Val / Test:")
print(evaluate_splits(model_tvm,
      {"Train": (X_train_full, y_train_full), "Val": (X_va, y_va),
       "Test":  (X_te, y_te)}).to_string(index=False))

m_opt = evaluate("TVM-LGB-Optuna-Fixed", y_te, proba_tvm)
print("\n[Key] CV AP ~ Test AP means Optuna is not overfit to the val set.")

In [ ]:
# CELL 9b -- TVM: Full algorithm comparison (LR / RF / XGB / CatBoost / LGB variants)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from catboost import CatBoostClassifier
import xgboost as xgb

_MC = ["split", "Accuracy", "AUC", "AP", "Precision", "Recall", "F1",
       "TP", "FP", "FN", "TN", "Brier"]

# -- Impute NaN for sklearn models -----------------------------------------------
# LGB / CatBoost / XGBoost handle NaN natively via their own split logic.
# LogisticRegression / RF / BalancedRF / RUSBoost do NOT -- they crash on NaN.
# Fit imputer on TRAIN only; transform val and test with the same statistics.
_imp_tvm = SimpleImputer(strategy="median")
X_tr_imp = pd.DataFrame(_imp_tvm.fit_transform(X_tr), columns=X_tr.columns)
X_va_imp = pd.DataFrame(_imp_tvm.transform(X_va),     columns=X_va.columns)
X_te_imp = pd.DataFrame(_imp_tvm.transform(X_te),     columns=X_te.columns)

tvm_splits_raw = {"Train": (X_tr,     y_tr), "Val": (X_va,     y_va), "Test": (X_te,     y_te)}
tvm_splits_imp = {"Train": (X_tr_imp, y_tr), "Val": (X_va_imp, y_va), "Test": (X_te_imp, y_te)}

# -- 1. Logistic Regression (impute -> scale -> fit) -----------------------------
print("--- 1. Logistic Regression ---")
scaler_tvm = StandardScaler()
Xtr_sc = scaler_tvm.fit_transform(X_tr_imp)
Xva_sc = scaler_tvm.transform(X_va_imp)
Xte_sc = scaler_tvm.transform(X_te_imp)
tvm_splits_lr = {"Train": (Xtr_sc, y_tr), "Val": (Xva_sc, y_va), "Test": (Xte_sc, y_te)}
lr_tvm = LogisticRegression(class_weight="balanced", C=0.1, max_iter=1000,
                             solver="lbfgs", random_state=42, n_jobs=-1)
lr_tvm.fit(Xtr_sc, y_tr)
proba_lr = lr_tvm.predict_proba(Xte_sc)[:, 1]
print(evaluate_splits(lr_tvm, tvm_splits_lr)[_MC].to_string(index=False))

# -- 2. Random Forest (imputed; NaN safe) ----------------------------------------
print("\n--- 2. Random Forest ---")
rf_tvm = RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=30,
                                 class_weight="balanced", random_state=42, n_jobs=-1)
rf_tvm.fit(X_tr_imp, y_tr)
proba_rf = rf_tvm.predict_proba(X_te_imp)[:, 1]
print(evaluate_splits(rf_tvm, tvm_splits_imp)[_MC].to_string(index=False))
print("[Note] RF tends to overfit (train AUC >> test) even with depth limit.")

# -- 3. XGBoost Baseline (NaN native) -------------------------------------------
print("\n--- 3. XGBoost ---")
xgb_tvm = xgb.XGBClassifier(
    n_estimators=1000, learning_rate=0.05, max_depth=6,
    scale_pos_weight=tvm_scale_pos_weight,
    random_state=42, n_jobs=-1, verbosity=0,
    eval_metric="auc", early_stopping_rounds=50)
xgb_tvm.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
proba_xgb = xgb_tvm.predict_proba(X_te)[:, 1]
print(evaluate_splits(xgb_tvm, tvm_splits_raw)[_MC].to_string(index=False))

# -- 4. CatBoost Baseline (NaN native) ------------------------------------------
print("\n--- 4. CatBoost ---")
cb_tvm = CatBoostClassifier(
    iterations=1000, learning_rate=0.05, depth=6,
    scale_pos_weight=tvm_scale_pos_weight,
    random_seed=42, verbose=0, early_stopping_rounds=50)
cb_tvm.fit(X_tr, y_tr, eval_set=(X_va, y_va))
proba_cb = cb_tvm.predict_proba(X_te)[:, 1]
print(evaluate_splits(cb_tvm, tvm_splits_raw)[_MC].to_string(index=False))

# -- 5. LGB Baseline (NaN native, from CELL 8) ----------------------------------
print("\n--- 5. LGB Baseline ---")
print(evaluate_splits(lgb_base_tvm, tvm_splits_raw)[_MC].to_string(index=False))

# -- 6. LGB Optuna-OLD (fixed val split -- shows the bug) -----------------------
def tvm_objective_old(trial):
    p = {"objective": "binary",
         "n_estimators"     : trial.suggest_int("n_estimators", 300, 1500),
         "learning_rate"    : trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
         "num_leaves"       : trial.suggest_int("num_leaves", 20, 150),
         "max_depth"        : trial.suggest_int("max_depth", 3, 10),
         "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
         "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.5, 1.0),
         "subsample"        : trial.suggest_float("subsample", 0.5, 1.0),
         "scale_pos_weight" : tvm_scale_pos_weight,
         "random_state": 42, "n_jobs": -1, "verbose": -1}
    m = lgb.LGBMClassifier(**p)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
          callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)])
    return roc_auc_score(y_va, m.predict_proba(X_va)[:, 1])

print("\n--- 6. LGB Optuna-OLD (fixed random val split -- 30 trials) ---")
study_tvm_old = optuna.create_study(direction="maximize")
study_tvm_old.optimize(tvm_objective_old, n_trials=30, show_progress_bar=True)
bp_old = study_tvm_old.best_params.copy()
bp_old.update({"objective": "binary", "scale_pos_weight": tvm_scale_pos_weight,
               "random_state": 42, "n_jobs": -1, "verbose": -1})
model_tvm_old = lgb.LGBMClassifier(**bp_old)
model_tvm_old.fit(X_tr, y_tr)
proba_tvm_old = model_tvm_old.predict_proba(X_te)[:, 1]
print(f"  Old best val AUC: {study_tvm_old.best_value:.4f}  <- inflated (fixed split)")
print(evaluate_splits(model_tvm_old, tvm_splits_raw)[_MC].to_string(index=False))

# -- 7. LGB Optuna-FIXED (TimeSeriesSplit, from CELL 9) -------------------------
print(f"\n--- 7. LGB Optuna-FIXED (TimeSeriesSplit, CV-AP={study_tvm.best_value:.4f}) ---")
print(evaluate_splits(model_tvm, {"Train": (X_train_full, y_train_full),
                                   "Val": (X_va, y_va), "Test": (X_te, y_te)})[_MC].to_string(index=False))

# -- MASTER TABLE models 1-7 ----------------------------------------------------
print("\n" + "=" * 108)
print("TVM -- MODELS 1-7  |  Test  |  threshold=0.50  (sorted AUC)")
print("=" * 108)
master = [
    evaluate("1-LR-Balanced",      y_te, proba_lr),
    evaluate("2-RF-Balanced",      y_te, proba_rf),
    evaluate("3-XGB",              y_te, proba_xgb),
    evaluate("4-CatBoost",         y_te, proba_cb),
    evaluate("5-LGB-Baseline",     y_te, lgb_base_tvm.predict_proba(X_te)[:, 1]),
    evaluate("6-LGB-Optuna-OLD",   y_te, proba_tvm_old),
    evaluate("7-LGB-Optuna-FIXED", y_te, proba_tvm),
]
print(pd.DataFrame(master).sort_values("AUC", ascending=False)[_MC].to_string(index=False))
print("\n[Workflow] Pick top 1-2 by Test AUC for Optuna. LR underperforms (linear boundary).")
print("[Note]     RF tends to overfit with class imbalance even with depth limit.")

# -- 8. BalancedRandomForest (imputed; NaN safe) ---------------------------------
print("\n--- 8. BalancedRandomForest (imbalance-aware bagging) ---")
from imblearn.ensemble import BalancedRandomForestClassifier
brf_tvm = BalancedRandomForestClassifier(
    n_estimators=300, max_depth=10, min_samples_leaf=20,
    sampling_strategy="auto", replacement=True,
    random_state=42, n_jobs=-1)
brf_tvm.fit(X_tr_imp, y_tr)
proba_brf_tvm = brf_tvm.predict_proba(X_te_imp)[:, 1]
print(evaluate_splits(brf_tvm, tvm_splits_imp)[_MC].to_string(index=False))
print("[Note] BalancedRF resamples majority class per tree -- no SMOTE distortion.")

# -- 9. RUSBoost (imputed; NaN safe) --------------------------------------------
print("\n--- 9. RUSBoost (random under-sampling boosted) ---")
from imblearn.ensemble import RUSBoostClassifier
rusb_tvm = RUSBoostClassifier(
    n_estimators=200, learning_rate=0.05, sampling_strategy="auto",
    random_state=42)
rusb_tvm.fit(X_tr_imp, y_tr)
proba_rusb_tvm = rusb_tvm.predict_proba(X_te_imp)[:, 1]
print(evaluate_splits(rusb_tvm, tvm_splits_imp)[_MC].to_string(index=False))

# -- FINAL MASTER TABLE all 9 models --------------------------------------------
print("\n" + "=" * 108)
print("TVM -- ALL 9 MODELS  |  Test  |  threshold=0.50  (sorted AUC)")
print("=" * 108)
master = [
    evaluate("1-LR-Balanced",      y_te, proba_lr),
    evaluate("2-RF-Balanced",      y_te, proba_rf),
    evaluate("3-XGB",              y_te, proba_xgb),
    evaluate("4-CatBoost",         y_te, proba_cb),
    evaluate("5-LGB-Baseline",     y_te, lgb_base_tvm.predict_proba(X_te)[:, 1]),
    evaluate("6-LGB-Optuna-OLD",   y_te, proba_tvm_old),
    evaluate("7-LGB-Optuna-FIXED", y_te, proba_tvm),
    evaluate("8-BalancedRF",       y_te, proba_brf_tvm),
    evaluate("9-RUSBoost",         y_te, proba_rusb_tvm),
]
print(pd.DataFrame(master).sort_values("AUC", ascending=False)[_MC].to_string(index=False))
print("\n[BalancedRF/RUSBoost] Best for extreme imbalance without SMOTE distribution shift.")
print("[Imputation note] Models 1,2,8,9 trained on median-imputed X. Models 3-7 use raw X (NaN-native).")

In [ ]:
# CELL 9c -- TVM: Ensemble Stacking (Level-0: XGB / CatBoost / RF / LGB)
# ─────────────────────────────────────────────────────────────────────────────
# PART 1: OOF loop (5-fold TimeSeriesSplit) — DIAGNOSTIC USE ONLY.
#   Models trained on partial data in early folds produce different probability
#   distributions than full-data models → do NOT use OOF probs for meta-learning.
#
# PART 2: HONEST val-set stacking.
#   xgb_tvm / cb_tvm / model_tvm use X_va for early stopping → val predictions
#   are contaminated (val AUC inflated 0.05-0.07 above test AUC).
#   Fix: retrain each with fixed n_estimators (from best_iteration_) and NO eval_set.
#   Same n_estimators on X_train_full → X_te for consistency between meta-train
#   and meta-test probability distributions.
#   RF is already clean (no early stopping) — used as-is.

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

_L0_TVM   = ["XGB", "CatBoost", "RF", "LGB"]
_tscv_stk = TimeSeriesSplit(n_splits=5)
_oof_tvm  = np.zeros((len(X_train_full), len(_L0_TVM)))

print("TVM Stacking -- OOF (5-fold TimeSeriesSplit) -- DIAGNOSTIC ...")
for _fi, (_tri, _vai) in enumerate(_tscv_stk.split(X_train_full)):
    _Xtr, _ytr = X_train_full.iloc[_tri], y_train_full.iloc[_tri]
    _Xva, _yva = X_train_full.iloc[_vai], y_train_full.iloc[_vai]
    _imp_f   = SimpleImputer(strategy="median")
    _Xtr_imp = pd.DataFrame(_imp_f.fit_transform(_Xtr), columns=_Xtr.columns)
    _Xva_imp = pd.DataFrame(_imp_f.transform(_Xva),     columns=_Xva.columns)

    _mx = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=6,
                             scale_pos_weight=tvm_scale_pos_weight,
                             eval_metric="auc", early_stopping_rounds=50,
                             random_state=42, n_jobs=-1, verbosity=0)
    _mx.fit(_Xtr, _ytr, eval_set=[(_Xva, _yva)], verbose=False)
    _oof_tvm[_vai, 0] = _mx.predict_proba(_Xva)[:, 1]

    _mc = CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=6,
                             scale_pos_weight=tvm_scale_pos_weight,
                             early_stopping_rounds=50, random_seed=42, verbose=0)
    _mc.fit(_Xtr, _ytr, eval_set=(_Xva, _yva))
    _oof_tvm[_vai, 1] = _mc.predict_proba(_Xva)[:, 1]

    _mr = RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=30,
                                 class_weight="balanced", random_state=42, n_jobs=-1)
    _mr.fit(_Xtr_imp, _ytr)
    _oof_tvm[_vai, 2] = _mr.predict_proba(_Xva_imp)[:, 1]

    _bp_oof = study_tvm.best_params.copy()
    _bp_oof.update({"objective": "binary", "scale_pos_weight": tvm_scale_pos_weight,
                    "n_estimators": 1000, "random_state": 42, "n_jobs": -1, "verbose": -1})
    _ml = lgb.LGBMClassifier(**_bp_oof)
    _ml.fit(_Xtr, _ytr, eval_set=[(_Xva, _yva)],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    _oof_tvm[_vai, 3] = _ml.predict_proba(_Xva)[:, 1]

    _fa = [roc_auc_score(_yva, _oof_tvm[_vai, i]) for i in range(len(_L0_TVM))]
    print(f"  Fold {_fi+1}: " + "  ".join(f"{n}={a:.4f}" for n, a in zip(_L0_TVM, _fa)))

print("\nOOF AUC overall (lower than fold AUC -- early-fold distribution shift):")
for _i, _n in enumerate(_L0_TVM):
    print(f"  {_n:<10}: {roc_auc_score(y_train_full, _oof_tvm[:, _i]):.4f}")

# ── PART 2: Honest stacking (retrain without eval_set) ───────────────────────
print("\n[Honest stacking] Retraining with fixed n_estimators (no eval_set) ...")

# Read actual stopping iterations from fitted models
# best_iteration_ = 0 when no early stopping was active; fall back to booster tree count
_xgb_n = int(getattr(xgb_tvm, "best_ntree_limit", 0) or
             (getattr(xgb_tvm, "best_iteration", 0) + 1) or 0)
if _xgb_n <= 0:
    try:    _xgb_n = xgb_tvm.get_booster().num_boosted_rounds()
    except: _xgb_n = 300
_xgb_n = max(_xgb_n, 1)

_cb_n = int(getattr(cb_tvm, "best_iteration_", 0) or 0)
if _cb_n <= 0:
    try:    _cb_n = int(cb_tvm.tree_count_)
    except: _cb_n = 300
_cb_n = max(_cb_n, 1)

_lgb_n = int(getattr(model_tvm, "best_iteration_", 0) or 0)
if _lgb_n <= 0:
    try:    _lgb_n = model_tvm.booster_.num_trees()
    except: _lgb_n = 300
_lgb_n = max(_lgb_n, 1)
print(f"  XGB={_xgb_n}  CB={_cb_n}  LGB={_lgb_n}  RF=already clean")

# Shared param dicts (no eval_set, no early_stopping_rounds)
_xgb_p = {"n_estimators": _xgb_n, "learning_rate": 0.05, "max_depth": 6,
           "scale_pos_weight": tvm_scale_pos_weight,
           "random_state": 42, "n_jobs": -1, "verbosity": 0}
_cb_p  = {"iterations": _cb_n, "learning_rate": 0.05, "depth": 6,
          "scale_pos_weight": tvm_scale_pos_weight,
          "random_seed": 42, "verbose": 0}
_lgb_p = {**study_tvm.best_params,
          "n_estimators": _lgb_n, "objective": "binary",
          "scale_pos_weight": tvm_scale_pos_weight,
          "random_state": 42, "n_jobs": -1, "verbose": -1}

# --- Retrain on X_tr (no eval_set) → honest X_va predictions ----------------
_xgb_tr = xgb.XGBClassifier(**_xgb_p);       _xgb_tr.fit(X_tr, y_tr)
_cb_tr  = CatBoostClassifier(**_cb_p);        _cb_tr.fit(X_tr, y_tr)
_lgb_tr = lgb.LGBMClassifier(**_lgb_p);       _lgb_tr.fit(X_tr, y_tr)

_va_stack_tvm = np.column_stack([
    _xgb_tr.predict_proba(X_va)[:, 1],
    _cb_tr.predict_proba(X_va)[:, 1],
    rf_tvm.predict_proba(X_va_imp)[:, 1],     # RF: always clean (no eval_set)
    _lgb_tr.predict_proba(X_va)[:, 1],
])

# Contamination check: honest val AUC should NOT exceed test AUC by > 0.02
_va_aucs_h = [roc_auc_score(y_va, _va_stack_tvm[:, i]) for i in range(4)]
_te_aucs_h = [roc_auc_score(y_te,
    np.array([xgb_tvm.predict_proba(X_te)[:, 1],
              cb_tvm.predict_proba(X_te)[:, 1],
              rf_tvm.predict_proba(X_te_imp)[:, 1],
              model_tvm.predict_proba(X_te)[:, 1]])[i]) for i in range(4)]
print(f"\nHonest val AUC vs test AUC (gap <= 0.02 = clean):")
for _n, _va, _te in zip(_L0_TVM, _va_aucs_h, _te_aucs_h):
    _gap = _va - _te
    _flag = "  OK" if _gap <= 0.03 else "  WARN (still contaminated)"
    print(f"  {_n:<10}: val={_va:.4f}  test={_te:.4f}  gap={_gap:+.4f}{_flag}")

# --- Retrain on X_train_full (no eval_set) → honest X_te predictions --------
print("  Retraining on X_train_full (no eval_set) for test predictions ...")
_xgb_tf = xgb.XGBClassifier(**_xgb_p);       _xgb_tf.fit(X_train_full, y_train_full)
_cb_tf  = CatBoostClassifier(**_cb_p);        _cb_tf.fit(X_train_full, y_train_full)
_lgb_tf = lgb.LGBMClassifier(**_lgb_p);       _lgb_tf.fit(X_train_full, y_train_full)

_te_stack_tvm = np.column_stack([
    _xgb_tf.predict_proba(X_te)[:, 1],
    _cb_tf.predict_proba(X_te)[:, 1],
    rf_tvm.predict_proba(X_te_imp)[:, 1],
    _lgb_tf.predict_proba(X_te)[:, 1],
])

print(f"\nHonest val-stack shape : {_va_stack_tvm.shape}")
print(f"Honest test-stack shape: {_te_stack_tvm.shape}")

# ── Ensemble A: Simple average ────────────────────────────────────────────────
proba_avg_tvm = _te_stack_tvm.mean(axis=1)

# ── Ensemble B: Val-AUC weighted average ─────────────────────────────────────
_w_va = np.array(_va_aucs_h, dtype=float)
_w_va = np.clip(_w_va - _w_va.min(), 0, None)
_w_va = _w_va / (_w_va.sum() + 1e-9)
proba_wavg_tvm = (_te_stack_tvm * _w_va).sum(axis=1)
print(f"\nVal-AUC weights (honest): " +
      "  ".join(f"{n}={w:.3f}" for n, w in zip(_L0_TVM, _w_va)))

# ── Ensemble C: Meta-LR (honest val-set, scaled) ─────────────────────────────
_meta_sc    = StandardScaler()
_va_stack_sc = _meta_sc.fit_transform(_va_stack_tvm)
_te_stack_sc  = _meta_sc.transform(_te_stack_tvm)

meta_lr_tvm = LogisticRegression(C=0.1, max_iter=2000, random_state=42, n_jobs=-1)
meta_lr_tvm.fit(_va_stack_sc, y_va)
proba_stk_tvm = meta_lr_tvm.predict_proba(_te_stack_sc)[:, 1]

_va_meta_auc = roc_auc_score(y_va, meta_lr_tvm.predict_proba(_va_stack_sc)[:, 1])
print(f"\nMeta-LR (honest) val AUC: {_va_meta_auc:.4f}  "
      f"(should be close to single-model val AUC ~{max(_va_aucs_h):.4f})")
print("Coefficients (positive = model adds value, should all be positive):")
for _n, _c in zip(_L0_TVM, meta_lr_tvm.coef_[0]):
    _sign_ok = "OK" if _c > 0 else "WARN (negative -- check contamination)"
    print(f"  {_n:<10}: {_c:+.4f}  [{_sign_ok}]")

# ── Ensemble D: Meta-LGB (honest val-set, scaled) ────────────────────────────
meta_lgb_tvm = lgb.LGBMClassifier(
    n_estimators=200, learning_rate=0.05, num_leaves=8, max_depth=3,
    scale_pos_weight=tvm_scale_pos_weight,
    min_child_samples=20, reg_lambda=5.0,
    random_state=42, n_jobs=-1, verbose=-1)
meta_lgb_tvm.fit(_va_stack_sc, y_va)
proba_meta_lgb_tvm = meta_lgb_tvm.predict_proba(_te_stack_sc)[:, 1]
print(f"Meta-LGB (honest) val AUC: "
      f"{roc_auc_score(y_va, meta_lgb_tvm.predict_proba(_va_stack_sc)[:, 1]):.4f}")

# ── Comparison table ──────────────────────────────────────────────────────────
print("\n" + "=" * 108)
print("TVM -- ENSEMBLE vs SINGLE MODELS  |  Test  |  threshold=0.50  (sorted AUC)")
print("=" * 108)
_rows = [
    evaluate("7-LGB-Optuna-FIXED",       y_te, proba_tvm),
    evaluate("4-CatBoost",               y_te, proba_cb),
    evaluate("3-XGB",                    y_te, proba_xgb),
    evaluate("A-SimpleAvg",              y_te, proba_avg_tvm),
    evaluate("B-ValAUC-WtdAvg",          y_te, proba_wavg_tvm),
    evaluate("C-MetaLR(honest,C=0.1)",   y_te, proba_stk_tvm),
    evaluate("D-MetaLGB(honest,d=3)",    y_te, proba_meta_lgb_tvm),
]
print(pd.DataFrame(_rows).sort_values("AUC", ascending=False)[_MC].to_string(index=False))

_ens_aucs = {
    "A-Avg"    : roc_auc_score(y_te, proba_avg_tvm),
    "B-WtdAvg" : roc_auc_score(y_te, proba_wavg_tvm),
    "C-MetaLR" : roc_auc_score(y_te, proba_stk_tvm),
    "D-MetaLGB": roc_auc_score(y_te, proba_meta_lgb_tvm),
}
_best_ens = max(_ens_aucs, key=_ens_aucs.get)
_single_auc = roc_auc_score(y_te, proba_tvm)
print(f"\n[Auto-select] Best ensemble: {_best_ens}  "
      f"(AUC={_ens_aucs[_best_ens]:.4f}  vs single LGB={_single_auc:.4f}  "
      f"delta={_ens_aucs[_best_ens] - _single_auc:+.4f})")
if _ens_aucs[_best_ens] > _single_auc + 0.002:
    print("  -> Ensemble adds value. Use for ranked dispatch.")
else:
    print("  -> Boosters are correlated. Ensemble does not improve over LGB-Optuna-FIXED.")
    print("     [Finding] 4 gradient boosting models trained on same temporal features")
    print("               converge to correlated predictions. This is expected.")
    print("     [Action]  Use LGB-Optuna-FIXED for ranked dispatch (Lift@5% > 2.0).")

In [ ]:
# CELL 9d - TVM: CatBoost Optuna (3-fold, 20 trials, timeout=600)
# Regularization levers:
#   l2_leaf_reg    -- L2 penalty on leaf values (CatBoost reg_lambda equivalent)
#   min_data_in_leaf -- structural regularisation; prevents tiny overfit leaves
#   random_strength  -- noise added to split scoring; discourages memorisation
#   colsample_bylevel -- per-depth-level feature subsampling

def tvm_cb_objective(trial):
    params = {
        "iterations"        : trial.suggest_int("iterations",          300,  800),
        "learning_rate"     : trial.suggest_float("learning_rate",    0.01, 0.15, log=True),
        "depth"             : trial.suggest_int("depth",                4,   10),
        "l2_leaf_reg"       : trial.suggest_float("l2_leaf_reg",       1.0, 30.0, log=True),
        "min_data_in_leaf"  : trial.suggest_int("min_data_in_leaf",    20,  200),
        "subsample"         : trial.suggest_float("subsample",          0.5,  1.0),
        "colsample_bylevel" : trial.suggest_float("colsample_bylevel",  0.5,  1.0),
        "random_strength"   : trial.suggest_float("random_strength",   0.1, 10.0, log=True),
        "scale_pos_weight"  : tvm_scale_pos_weight,
        "random_seed": 42, "verbose": 0,
    }
    aucs = []
    for _tri, _vai in tscv_opt.split(X_train_full):   # 3-fold from Cell 9
        _m = CatBoostClassifier(**params, early_stopping_rounds=20)
        _m.fit(X_train_full.iloc[_tri], y_train_full.iloc[_tri],
               eval_set=(X_train_full.iloc[_vai], y_train_full.iloc[_vai]),
               verbose=False)
        aucs.append(roc_auc_score(y_train_full.iloc[_vai],
                                  _m.predict_proba(X_train_full.iloc[_vai])[:, 1]))
    return float(np.mean(aucs))

print("Running Optuna CatBoost TVM -- 3-fold x 20 trials, timeout=600s ...")
study_tvm_cb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=1),
)
study_tvm_cb.optimize(tvm_cb_objective, n_trials=20, timeout=600,
                      show_progress_bar=True)

best_cb_p = study_tvm_cb.best_params.copy()
best_cb_p.update({
    "scale_pos_weight": tvm_scale_pos_weight,
    "random_seed": 42, "verbose": 0, "early_stopping_rounds": 30,
})
model_tvm_cb = CatBoostClassifier(**best_cb_p)
model_tvm_cb.fit(X_train_full, y_train_full, eval_set=(X_va, y_va), verbose=False)
proba_tvm_cb = model_tvm_cb.predict_proba(X_te)[:, 1]

print(f"\nCatBoost Optuna CV AUC : {study_tvm_cb.best_value:.4f}")
print(f"LGB     Optuna CV AP   : {study_tvm.best_value:.4f}")

print(f"\nBest CatBoost params:")
print(f"  depth={best_cb_p.get('depth')}  l2_leaf_reg={best_cb_p.get('l2_leaf_reg',0):.2f}"
      f"  min_data_in_leaf={best_cb_p.get('min_data_in_leaf')}")

print("\nLGB vs CatBoost (Test, sorted by AUC):")
_mc = ["split","AUC","AP","Precision","Recall","F1","TP","FP","FN","Brier"]
print(pd.DataFrame([
    evaluate("LGB-Optuna-Fixed", y_te, proba_tvm),
    evaluate("CB-Optuna-Fixed",  y_te, proba_tvm_cb),
]).sort_values("AUC", ascending=False)[_mc].to_string(index=False))
print("\n[Best] Use whichever is higher here for stacking (Cell 9c).")

In [ ]:
# CELL 9e -- TVM: Probability calibration (isotonic, val-set holdout)
# ─────────────────────────────────────────────────────────────────────────────
# Problem: LGB trained on imbalanced data (~6% positives) produces probabilities
#   that cluster near 0. p=0.15 could actually mean "40% of this risk tier fails".
#   Threshold selection becomes arbitrary and not reproducible across retrains.
#
# Fix: CalibratedClassifierCV(prefit=True) learns a monotone mapping from
#   raw LGB scores → calibrated probabilities using the val set as a holdout.
#   method='isotonic' (non-parametric, piecewise constant) outperforms 'sigmoid'
#   when the score-to-probability mapping is not log-linear.
#   prefit=True: base model already trained; calibrator is fitted on X_va.
#
# AUC is preserved (calibration is a monotone transform of the score).
# AP changes slightly because it depends on the probability values, not just rank.

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
import matplotlib.pyplot as plt

# ── Calibrate on val set ──────────────────────────────────────────────────────
_calib_cv = CalibratedClassifierCV(model_tvm, method="isotonic", cv="prefit")
_calib_cv.fit(X_va, y_va)
proba_calib_tvm = _calib_cv.predict_proba(X_te)[:, 1]

# ── Reliability diagrams ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for _ax, _prob, _lbl in [
    (axes[0], proba_tvm,       "Uncalibrated LGB"),
    (axes[1], proba_calib_tvm, "Calibrated (isotonic, val-set)"),
]:
    _fp, _mp = calibration_curve(y_te, _prob, n_bins=10, strategy="quantile")
    _ax.plot(_mp, _fp, "o-", color="#2196F3", label="Model")
    _ax.plot([0, 1], [0, 1], "k--", alpha=0.6, label="Perfect")
    _ax.set_xlabel("Mean predicted probability")
    _ax.set_ylabel("Fraction of positives")
    _ax.set_title(_lbl)
    _ax.legend()
    _ax.grid(True, alpha=0.3)
    _ax.set_xlim(-0.02, 1.02)
    _ax.set_ylim(-0.02, 1.02)
plt.suptitle("TVM — Calibration Curves (quantile bins)", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

# ── Metric comparison ─────────────────────────────────────────────────────────
print("TVM — Uncalibrated vs Calibrated (threshold=0.50)")
print("=" * 72)
for _lbl, _p in [("Uncalibrated", proba_tvm), ("Calibrated  ", proba_calib_tvm)]:
    _r = evaluate(f"TVM ({_lbl.strip()})", y_te, _p, threshold=0.50)
    _brier = float(np.mean((_p - y_te.values) ** 2))
    print(f"  {_lbl}: AUC={_r['AUC']:.4f}  AP={_r['AP']:.4f}  Brier={_brier:.4f}  "
          f"proba_mean={_p.mean():.4f}  proba_p95={np.percentile(_p, 95):.4f}")

print(f"\n[Note] AUC is preserved (calibration is a monotone score transform).")
print(f"[Note] Use proba_calib_tvm in Cell 10 for threshold tuning —")
print(f"       probabilities now approximate P(failure in 3d | features).")
print(f"[Note] Brier score measures mean-squared probability error; lower is better.")

In [ ]:
# CELL 10 -- TVM threshold tuning (F2-optimal + budget-constrained + recall-SLA)
# ─────────────────────────────────────────────────────────────────────────────
# Three operationally meaningful thresholds:
#   opt_thresh_tvm     : F2-optimal (beta=2: recall 4x more than precision)
#   budget_thresh_tvm  : flags top DISPATCH_BUDGET_PCT of fleet per day  ← NEW
#   recall_thresh_tvm  : highest threshold where recall >= RECALL_FLOOR
#
# Budget threshold: in practice, maintenance teams can service K devices/day.
# Set DISPATCH_BUDGET_PCT to match team capacity. This gives the threshold that
# exactly fills the team's daily capacity with the highest-risk devices.
# Overrides F2 when F2-optimal coverage > 40% of fleet (impractical).

from sklearn.metrics import precision_recall_curve as _prc

DISPATCH_BUDGET_PCT = 0.25   # flag top 25% of fleet per day (adjust to team capacity)

# Probability source: calibrated if Cell 9e ran, else raw
try:
    _proba_th = proba_calib_tvm
    _proba_src = "calibrated (proba_calib_tvm)"
except NameError:
    _proba_th = proba_tvm
    _proba_src = "uncalibrated (proba_tvm)"
print(f"Threshold source: {_proba_src}")
print(f"Dispatch budget : top {DISPATCH_BUDGET_PCT:.0%} of fleet per day")

# ── F2 threshold ──────────────────────────────────────────────────────────────
_prec_arr, _rec_arr, _thr_arr = _prc(y_te, _proba_th)
_f2_vals  = (5 * _prec_arr[:-1] * _rec_arr[:-1]) / (4 * _prec_arr[:-1] + _rec_arr[:-1] + 1e-9)
_best_idx = int(np.argmax(_f2_vals))
opt_thresh_tvm  = float(_thr_arr[_best_idx])
opt_prec_tvm    = float(_prec_arr[_best_idx])
opt_rec_tvm     = float(_rec_arr[_best_idx])
opt_f2_tvm      = float(_f2_vals[_best_idx])
_fl_f2          = float((_proba_th >= opt_thresh_tvm).mean())
_f2_orig_thresh = opt_thresh_tvm  # save BEFORE auto-guard may override opt_thresh_tvm

# ── Budget-constrained threshold ──────────────────────────────────────────────
# Threshold = the DISPATCH_BUDGET_PCT-th highest predicted probability.
# Devices above this threshold are ranked top-K for dispatch.
_n_budget = int(DISPATCH_BUDGET_PCT * len(_proba_th))
_sorted_desc = np.sort(_proba_th)[::-1]
budget_thresh_tvm = float(_sorted_desc[_n_budget]) if _n_budget < len(_sorted_desc) else float(_sorted_desc[-1])
_budget_prec = float(y_te.values[_proba_th >= budget_thresh_tvm].mean())
_budget_rec  = float((y_te.values[_proba_th >= budget_thresh_tvm]).sum() / y_te.sum())

# ── Recall-SLA threshold ──────────────────────────────────────────────────────
_viable = _thr_arr[_rec_arr[:-1] >= QUALITY_RECALL_FLOOR["TVM"]]
recall_thresh_tvm = float(_viable[-1]) if len(_viable) > 0 else float(_thr_arr[0])
_fl_sla           = float((_proba_th >= recall_thresh_tvm).mean())

# ── Threshold summary ─────────────────────────────────────────────────────────
print(f"\nThreshold summary:")
print(f"  F2-optimal    : {opt_thresh_tvm:.4f}  prec={opt_prec_tvm:.3f}  rec={opt_rec_tvm:.3f}  "
      f"F2={opt_f2_tvm:.4f}  fleet={_fl_f2:.1%}")
print(f"  Budget ({DISPATCH_BUDGET_PCT:.0%} fleet): {budget_thresh_tvm:.4f}  prec={_budget_prec:.3f}  rec={_budget_rec:.3f}  "
      f"                fleet={DISPATCH_BUDGET_PCT:.1%}")
print(f"  Recall-SLA    : {recall_thresh_tvm:.4f}  "
      f"(recall >= {QUALITY_RECALL_FLOOR['TVM']:.0%})  fleet={_fl_sla:.1%}")

# Auto-select operational threshold
if _fl_f2 > 0.40:
    _op_thresh = budget_thresh_tvm
    _op_label  = f"budget ({DISPATCH_BUDGET_PCT:.0%})"
    print(f"\n  [Auto] F2-optimal flags {_fl_f2:.0%} of fleet -- too wide for dispatch.")
    print(f"         Switching to budget threshold: {budget_thresh_tvm:.4f} ({DISPATCH_BUDGET_PCT:.0%} fleet)")
else:
    _op_thresh = opt_thresh_tvm
    _op_label  = "F2-optimal"
    print(f"\n  [Auto] F2-optimal flags {_fl_f2:.0%} of fleet -- within operational range.")

# Override opt_thresh_tvm with the selected operational threshold
opt_thresh_tvm = _op_thresh
print(f"  opt_thresh_tvm set to: {opt_thresh_tvm:.4f} ({_op_label})")

# ── Threshold sweep ───────────────────────────────────────────────────────────
print("\nThreshold sweep:")
print(threshold_sweep(y_te, _proba_th).to_string(index=False))

# ── Side-by-side comparison ───────────────────────────────────────────────────
# Always show original F2 threshold (before auto-guard override) so table is never duplicated
_bgt_tag = " [OPERATIONAL]" if abs(opt_thresh_tvm - budget_thresh_tvm) < 1e-8 else ""
_m_f2  = evaluate(f"TVM @ {_f2_orig_thresh:.4f} (F2-opt, fleet={_fl_f2:.0%})",
                  y_te, _proba_th, threshold=_f2_orig_thresh)
_m_bgt = evaluate(f"TVM @ {budget_thresh_tvm:.4f} (budget {DISPATCH_BUDGET_PCT:.0%}){_bgt_tag}",
                  y_te, _proba_th, threshold=budget_thresh_tvm)
_m_sla = evaluate(f"TVM @ {recall_thresh_tvm:.4f} (recall-SLA)",
                  y_te, _proba_th, threshold=recall_thresh_tvm)
_m_def = evaluate("TVM @ 0.50",
                  y_te, _proba_th, threshold=0.50)
print("\n[Threshold comparison]")
print(pd.DataFrame([_m_f2, _m_bgt, _m_sla, _m_def]).to_string(index=False))

# ── Operational guidance ──────────────────────────────────────────────────────
print(f"\n[Operational guidance]")
print(f"  Budget threshold {budget_thresh_tvm:.4f}: flags {DISPATCH_BUDGET_PCT:.0%} fleet/day  "
      f"(~{_n_budget} devices)  -- RECOMMENDED default")
print(f"  Adjust DISPATCH_BUDGET_PCT (currently {DISPATCH_BUDGET_PCT:.0%}) to match daily team capacity.")
print(f"  Best practice: rank devices by {_proba_src.split()[0]} probability; dispatch top-{_n_budget}.")

plot_eval("TVM -- Best Model (Optuna Fixed)", y_te, _proba_th,
          threshold=opt_thresh_tvm, save_path=f"{EVAL_DIR}/tvm_eval.png")

In [ ]:
# CELL 10b — TVM: Ranking metrics (precision@K, recall@K, lift@K)
# Operations teams dispatch maintenance from a ranked list of highest-risk devices.
# These metrics measure how many true failures appear near the top of that list.
# Target: lift@5% > 2.0 means the model ranks failures 2x above random inspection.

print_ranking_table("TVM", y_te, proba_tvm)

# Also show for the stacking ensemble if it beat the single model
try:
    _stk_ap = average_precision_score(y_te, proba_stk_tvm)
    _tvm_ap  = average_precision_score(y_te, proba_tvm)
    if _stk_ap > _tvm_ap:
        print()
        print_ranking_table("TVM Stacking (better AP)", y_te, proba_stk_tvm)
except NameError:
    pass  # stacking cell not yet run

In [ ]:
# CELL 10c -- TVM: Temporal drift check (monthly AUC on test set)
# ─────────────────────────────────────────────────────────────────────────────
# Splits the test set into monthly windows and reports AUC + AP per window.
# A sustained AUC drop > 0.05 signals concept drift — schedule model retraining.
# Shows both raw and calibrated AUC when Cell 9e has run.

import matplotlib.pyplot as plt

_has_calib = "proba_calib_tvm" in dir() and proba_calib_tvm is not None

_drift_df_input = pd.DataFrame({
    "_date": pd.to_datetime(test_tvm[DATE_COL].values),
    "y_true": y_te.values,
    "y_raw":  proba_tvm,
})
if _has_calib:
    _drift_df_input["y_cal"] = proba_calib_tvm

_drift_df_input["ym"] = _drift_df_input["_date"].dt.to_period("M")

_windows = []
for _ym, _g in _drift_df_input.groupby("ym"):
    if int(_g["y_true"].sum()) < 5:
        continue   # skip windows with too few positives (unstable AUC)
    _row = {
        "period"   : str(_ym),
        "n"        : len(_g),
        "positives": int(_g["y_true"].sum()),
        "pos_rate" : round(_g["y_true"].mean(), 4),
        "AUC_raw"  : round(roc_auc_score(_g["y_true"], _g["y_raw"]), 4),
        "AP_raw"   : round(average_precision_score(_g["y_true"], _g["y_raw"]), 4),
    }
    if _has_calib:
        _row["AUC_cal"] = round(roc_auc_score(_g["y_true"], _g["y_cal"]), 4)
        _row["AP_cal"]  = round(average_precision_score(_g["y_true"], _g["y_cal"]), 4)
    _windows.append(_row)

_drift_tbl = pd.DataFrame(_windows)
print("TVM — Temporal drift check (monthly test windows, min 5 positives)")
print("=" * 82)
if len(_drift_tbl) == 0:
    print("  [Skip] No monthly windows with >= 5 positives in test set.")
    print("         Test period may be too short or positive rate too low.")
else:
    print(_drift_tbl.to_string(index=False))

    # Drift summary
    _first_auc = _drift_tbl["AUC_raw"].iloc[0]
    _last_auc  = _drift_tbl["AUC_raw"].iloc[-1]
    _min_auc   = _drift_tbl["AUC_raw"].min()
    _delta     = _last_auc - _first_auc
    print(f"\n[Drift summary]  first={_first_auc:.4f}  last={_last_auc:.4f}  "
          f"min={_min_auc:.4f}  delta={_delta:+.4f}")
    if _delta < -0.05:
        print("  STATUS: WARNING — AUC dropped > 0.05 from first to last window.")
        print("          Recommend: schedule retraining on extended data window.")
    elif _delta < -0.02:
        print("  STATUS: INFO — mild drift detected. Monitor next 1-2 months.")
    else:
        print("  STATUS: OK — no significant drift across test windows.")
    print(f"\n  Retraining trigger: if monthly AUC < {max(0.50, _min_auc - 0.05):.3f}  "
          f"(min_seen - 0.05), retrain on expanded dataset.")

    # Plot
    if len(_drift_tbl) >= 2:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
        _x = list(range(len(_drift_tbl)))
        _xlbl = _drift_tbl["period"].tolist()

        # AUC over time
        ax1.plot(_x, _drift_tbl["AUC_raw"], "o-", color="#2196F3", label="AUC raw")
        if "AUC_cal" in _drift_tbl.columns:
            ax1.plot(_x, _drift_tbl["AUC_cal"], "s--", color="#4CAF50", label="AUC calibrated")
        ax1.axhline(0.5, color="#9E9E9E", linestyle=":", linewidth=0.8)
        ax1.axhline(_drift_tbl["AUC_raw"].mean(), color="#FF9800", linestyle="--",
                    linewidth=0.9, label=f"Mean={_drift_tbl['AUC_raw'].mean():.3f}")
        ax1.set_xticks(_x); ax1.set_xticklabels(_xlbl, rotation=30, ha="right")
        ax1.set_ylabel("AUC"); ax1.set_title("TVM — AUC by month (drift check)")
        ax1.legend(); ax1.grid(True, alpha=0.3)

        # AP + positive rate over time
        _ax2b = ax2.twinx()
        ax2.bar(_x, _drift_tbl["pos_rate"], alpha=0.3, color="#BDBDBD", label="Pos rate")
        _ax2b.plot(_x, _drift_tbl["AP_raw"], "o-", color="#E91E63", label="AP raw")
        if "AP_cal" in _drift_tbl.columns:
            _ax2b.plot(_x, _drift_tbl["AP_cal"], "s--", color="#9C27B0", label="AP calibrated")
        ax2.set_xticks(_x); ax2.set_xticklabels(_xlbl, rotation=30, ha="right")
        ax2.set_ylabel("Positive rate", color="#616161")
        _ax2b.set_ylabel("AP")
        ax2.set_title("TVM — AP + Positive Rate by month")
        lines1, labs1 = ax2.get_legend_handles_labels()
        lines2, labs2 = _ax2b.get_legend_handles_labels()
        ax2.legend(lines1 + lines2, labs1 + labs2, loc="upper right")
        ax2.grid(True, alpha=0.3)

        plt.suptitle("TVM — Temporal Drift Check", y=1.02, fontsize=13)
        plt.tight_layout()
        plt.show()
    else:
        print("  [Plot skipped] Need >= 2 windows.")

In [ ]:
# CELL 11 — TVM SHAP feature importance (beeswarm + bar + FN vs TP suppression)
print("Computing TVM SHAP values (sample 2000 rows) ...")
_sample_size = min(2000, len(X_te))
_Xte_shap = X_te.iloc[:_sample_size]
explainer_tvm = shap.TreeExplainer(model_tvm)
shap_tvm = explainer_tvm.shap_values(_Xte_shap)

# ── Beeswarm (value distribution per feature) ─────────────────────────────────
plt.figure()
shap.summary_plot(shap_tvm, _Xte_shap, feature_names=FEATURE_COLS,
                  max_display=25, show=False)
plt.title("TVM — SHAP Beeswarm (value distribution)", fontsize=12)
plt.tight_layout()
plt.savefig(f"{EVAL_DIR}/tvm_shap_beeswarm.png", dpi=120, bbox_inches="tight")
plt.show()

# ── Bar plot (mean |SHAP| importance) ─────────────────────────────────────────
plt.figure()
shap.summary_plot(shap_tvm, _Xte_shap, feature_names=FEATURE_COLS,
                  plot_type="bar", max_display=25, show=False)
plt.title("TVM — SHAP Bar (mean |SHAP|)", fontsize=12)
plt.tight_layout()
plt.savefig(f"{EVAL_DIR}/tvm_shap_bar.png", dpi=120, bbox_inches="tight")
plt.show()

# ── FN vs TP suppression analysis ─────────────────────────────────────────────
# For failed predictions (FN) vs correctly caught failures (TP):
# which features have the lowest SHAP magnitude on missed failures?
# High suppression = signal exists but model ignores it on misses.
_y_pred_tvm = (proba_tvm[:_sample_size] >= opt_thresh_tvm).astype(int)
_y_true_tvm = np.array(y_te)[:_sample_size]
_tp_idx = np.where((_y_pred_tvm == 1) & (_y_true_tvm == 1))[0]
_fn_idx = np.where((_y_pred_tvm == 0) & (_y_true_tvm == 1))[0]

print(f"\nTVM — SHAP FN vs TP analysis  (TPs={len(_tp_idx)}, FNs={len(_fn_idx)})")
if len(_tp_idx) >= 5 and len(_fn_idx) >= 5:
    _shap_arr = np.array(shap_tvm)
    _shap_tp  = np.abs(_shap_arr[_tp_idx]).mean(axis=0)
    _shap_fn  = np.abs(_shap_arr[_fn_idx]).mean(axis=0)
    _n_feats  = min(len(FEATURE_COLS), len(_shap_tp))
    _supp_df  = pd.DataFrame({
        "feature"    : FEATURE_COLS[:_n_feats],
        "shap_tp"    : _shap_tp[:_n_feats],
        "shap_fn"    : _shap_fn[:_n_feats],
        "suppression": _shap_tp[:_n_feats] - _shap_fn[:_n_feats],
    }).sort_values("suppression", ascending=False)
    print("Top 15 suppressed features (high shap on TP, low on FN — model blind spots):")
    print(_supp_df.head(15)[["feature","shap_tp","shap_fn","suppression"]].to_string(index=False))
    print("\n[Interpret] High suppression on key feature = consider adding interaction")
    print("            term or feature cross to make it more visible on missed failures.")
else:
    print(f"  [Suppression] Too few TPs ({len(_tp_idx)}) or FNs ({len(_fn_idx)}) "
          f"— lower opt_thresh_tvm or use val set.")

---
## GATE — Evaluation & Fix
**Issues:**
- Extreme imbalance: 0.63% positive rate → scale_pos_weight=157 → AP only 0.04–0.07.
- RF train F1=0.82, test F1=0.13 — pure overfit; dropped.
- Default threshold=0.5 catches only 7–16% of failures.

**Fixes:**
1. Moderate SMOTE: oversample positives to 5% (less aggressive than full rebalancing).
2. Threshold sweep to find the recall≥55% operating point.
3. Calibration curve to assess probability reliability.

In [ ]:
# CELL 12 - GATE data split + imbalance diagnosis
# Uses FEATURE_COLS_GATE (zero-MI + collinear features for GATE removed in Cell 5b).
# If clean label recomputation succeeded in CELL 4b, uses will_fail_3d_clean.
_gate_target = "will_fail_3d_clean" if _USE_CLEAN_LABEL_GATE else TARGET_COL
print(f"GATE target column   : {_gate_target}")
print(f"GATE feature count   : {len(FEATURE_COLS_GATE)}  "
      f"(TVM has {len(FEATURE_COLS_TVM)})")

train_gate, val_gate, test_gate = time_split(
    df_gate,
    train_end=TRAIN_END_DATE, val_end=VAL_END_DATE, embargo_days=EMBARGO_DAYS,
)

X_tr_g, y_tr_g = train_gate[FEATURE_COLS_GATE], train_gate[_gate_target]
X_va_g, y_va_g = val_gate[FEATURE_COLS_GATE],   val_gate[_gate_target]
X_te_g, y_te_g = test_gate[FEATURE_COLS_GATE],  test_gate[_gate_target]

gate_pos_rate = y_tr_g.mean()
print(f"\nGATE Train : {len(X_tr_g):,}  (pos={y_tr_g.sum():,}  {gate_pos_rate:.2%})")
print(f"GATE Val   : {len(X_va_g):,}  (pos={y_va_g.sum():,}  {y_va_g.mean():.2%})")
print(f"GATE Test  : {len(X_te_g):,}  (pos={y_te_g.sum():,}  {y_te_g.mean():.2%})")
print(f"\nNaive scale_pos_weight (raw): {(1-gate_pos_rate)/gate_pos_rate:.0f}  <- extreme")
print("Fix: moderate SMOTE to 5%, then scale_pos_weight = (1-0.05)/0.05 = 19")

print(f"\nDate range -- Train : {train_gate[DATE_COL].min().date()} -> "
      f"{train_gate[DATE_COL].max().date()}")
print(f"Date range -- Val   : {val_gate[DATE_COL].min().date()} -> "
      f"{val_gate[DATE_COL].max().date()}")
print(f"Date range -- Test  : {test_gate[DATE_COL].min().date()} -> "
      f"{test_gate[DATE_COL].max().date()}")

In [ ]:
# CELL 13 — GATE FIX: Impute + Moderate SMOTE (5%) + LGB

from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
import lightgbm as lgb

SMOTE_RATIO = 0.05

# 1) Impute missing values before SMOTE
imputer_g = SimpleImputer(strategy="median")

X_tr_g_imp = imputer_g.fit_transform(X_tr_g)
X_va_g_imp = imputer_g.transform(X_va_g)
X_te_g_imp = imputer_g.transform(X_te_g)

# 2) Apply SMOTE only on training data
smote = SMOTE(
    sampling_strategy=SMOTE_RATIO,
    random_state=42
)

X_tr_g_sm, y_tr_g_sm = smote.fit_resample(X_tr_g_imp, y_tr_g)

smote_pos_rate = y_tr_g_sm.mean()
smote_spw = (1 - smote_pos_rate) / smote_pos_rate

print(f"After SMOTE: {len(X_tr_g_sm):,} rows | pos rate: {smote_pos_rate:.2%}")
print(f"scale_pos_weight (post-SMOTE): {smote_spw:.1f}")

# 3) Train LGBM on imputed + SMOTE data
lgb_gate = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=7,
    min_child_samples=100,
    colsample_bytree=0.8,
    subsample=0.8,
    subsample_freq=1,
    scale_pos_weight=smote_spw,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgb_gate.fit(
    X_tr_g_sm,
    y_tr_g_sm,
    eval_set=[(X_va_g_imp, y_va_g)],
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(period=200)
    ]
)

proba_gate = lgb_gate.predict_proba(X_te_g_imp)[:, 1]

print("\nGATE LGB — Train(SMOTE) / Val / Test (threshold=0.50):")

gate_splits = {
    "Train(SMOTE)": (X_tr_g_sm, y_tr_g_sm),
    "Val": (X_va_g_imp, y_va_g),
    "Test": (X_te_g_imp, y_te_g)
}

print(
    evaluate_splits(
        lgb_gate,
        gate_splits,
        threshold=0.50
    ).to_string(index=False)
)

print("[Note] Train row reflects SMOTE-augmented distribution; Val/Test are raw but median-imputed.")
print("[Interpret] High Accuracy with near-zero Recall at t=0.50 = imbalance masking; use threshold sweep.")

m_gate_base = evaluate(
    "GATE-LGB-Base",
    y_te_g,
    proba_gate
)


In [ ]:
# CELL 13b — GATE: Full algorithm comparison
# LR / RF / XGB / CatBoost / LGB variants
# FIXED: use imputed NumPy arrays consistently for SMOTE models

import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier
import xgboost as xgb
import lightgbm as lgb
import optuna

_cols_comp = [
    "split", "Accuracy", "AUC", "AP", "Precision", "Recall", "F1",
    "TP", "FP", "FN", "TN", "Brier"
]

gate_raw_spw = (1 - gate_pos_rate) / gate_pos_rate

# ============================================================
# IMPORTANT:
# These should already exist from Cell 13:
# X_tr_g_imp, X_va_g_imp, X_te_g_imp
# X_tr_g_sm, y_tr_g_sm
# smote_spw
# lgb_gate
# proba_gate
# ============================================================

gate_sm_splits = {
    "Train(SMOTE)": (X_tr_g_sm, y_tr_g_sm),
    "Val": (X_va_g_imp, y_va_g),
    "Test": (X_te_g_imp, y_te_g)
}

# ============================================================
# 1. Logistic Regression
# Raw train + scaling + median imputation + class_weight
# ============================================================

print("--- 1. Logistic Regression (raw train, class_weight=balanced) ---")

scaler_gate = StandardScaler()
Xtr_g_sc = scaler_gate.fit_transform(X_tr_g)
Xva_g_sc = scaler_gate.transform(X_va_g)
Xte_g_sc = scaler_gate.transform(X_te_g)

imputer_lr_gate = SimpleImputer(strategy="median")
Xtr_g_sc = imputer_lr_gate.fit_transform(Xtr_g_sc)
Xva_g_sc = imputer_lr_gate.transform(Xva_g_sc)
Xte_g_sc = imputer_lr_gate.transform(Xte_g_sc)

lr_gate = LogisticRegression(
    class_weight="balanced",
    C=0.1,
    max_iter=1000,
    solver="lbfgs",
    random_state=42,
    n_jobs=-1
)

lr_gate.fit(Xtr_g_sc, y_tr_g)

proba_lr_train = lr_gate.predict_proba(Xtr_g_sc)[:, 1]
proba_lr_val = lr_gate.predict_proba(Xva_g_sc)[:, 1]
proba_lr_gate = lr_gate.predict_proba(Xte_g_sc)[:, 1]

lr_gate_rows = [
    evaluate("Train", y_tr_g, proba_lr_train),
    evaluate("Val", y_va_g, proba_lr_val),
    evaluate("Test", y_te_g, proba_lr_gate)
]

print(pd.DataFrame(lr_gate_rows)[_cols_comp].to_string(index=False))


# ============================================================
# 2. Random Forest
# SMOTE train + imputed val/test
# ============================================================

print("\n--- 2. Random Forest (SMOTE train, class_weight=balanced) ---")

rf_gate = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=30,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_gate.fit(X_tr_g_sm, y_tr_g_sm)

proba_rf_gate = rf_gate.predict_proba(X_te_g_imp)[:, 1]

print(
    evaluate_splits(
        rf_gate,
        gate_sm_splits,
        threshold=0.50
    )[_cols_comp].to_string(index=False)
)


# ============================================================
# 3. XGBoost
# SMOTE train + imputed val/test
# ============================================================

print("\n--- 3. XGBoost (SMOTE train) ---")

xgb_gate = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=7,
    scale_pos_weight=smote_spw,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
    eval_metric="auc",
    early_stopping_rounds=50
)

xgb_gate.fit(
    X_tr_g_sm,
    y_tr_g_sm,
    eval_set=[(X_va_g_imp, y_va_g)],
    verbose=False
)

proba_gate_xgb = xgb_gate.predict_proba(X_te_g_imp)[:, 1]

print(
    evaluate_splits(
        xgb_gate,
        gate_sm_splits,
        threshold=0.50
    )[_cols_comp].to_string(index=False)
)


# ============================================================
# 4. CatBoost
# SMOTE train + imputed val/test
# ============================================================

print("\n--- 4. CatBoost (SMOTE train) ---")

cb_gate = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=7,
    scale_pos_weight=smote_spw,
    random_seed=42,
    verbose=0,
    early_stopping_rounds=50
)

cb_gate.fit(
    X_tr_g_sm,
    y_tr_g_sm,
    eval_set=(X_va_g_imp, y_va_g)
)

proba_cb_gate = cb_gate.predict_proba(X_te_g_imp)[:, 1]

print(
    evaluate_splits(
        cb_gate,
        gate_sm_splits,
        threshold=0.50
    )[_cols_comp].to_string(index=False)
)


# ============================================================
# 5. LGB No-SMOTE
# Raw train with raw extreme scale_pos_weight
# LightGBM can handle NaN, but for consistency we use imputed data
# ============================================================

print(f"\n--- 5. LGB No-SMOTE (raw scale_pos_weight={gate_raw_spw:.0f}) ---")

lgb_gate_raw = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=7,
    min_child_samples=100,
    colsample_bytree=0.8,
    subsample=0.8,
    subsample_freq=1,
    scale_pos_weight=gate_raw_spw,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgb_gate_raw.fit(
    X_tr_g_imp,
    y_tr_g,
    eval_set=[(X_va_g_imp, y_va_g)],
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(0)
    ]
)

proba_gate_raw = lgb_gate_raw.predict_proba(X_te_g_imp)[:, 1]

gate_raw_splits = {
    "Train": (X_tr_g_imp, y_tr_g),
    "Val": (X_va_g_imp, y_va_g),
    "Test": (X_te_g_imp, y_te_g)
}

print(
    evaluate_splits(
        lgb_gate_raw,
        gate_raw_splits,
        threshold=0.50
    )[_cols_comp].to_string(index=False)
)


# ============================================================
# 6. LGB Optuna-OLD
# Fixed validation split + SMOTE
# ============================================================

def gate_objective_old(trial):

    params = {
        "objective": "binary",
        "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 100),
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "min_child_samples": trial.suggest_int("min_child_samples", 50, 300),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq": 1,
        "scale_pos_weight": smote_spw,
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 2.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1
    }

    m = lgb.LGBMClassifier(**params)

    m.fit(
        X_tr_g_sm,
        y_tr_g_sm,
        eval_set=[(X_va_g_imp, y_va_g)],
        callbacks=[
            lgb.early_stopping(30, verbose=False),
            lgb.log_evaluation(0)
        ]
    )

    val_proba = m.predict_proba(X_va_g_imp)[:, 1]

    return roc_auc_score(y_va_g, val_proba)


print("\n--- 6. LGB Optuna-OLD (fixed val split + SMOTE — 30 trials) ---")

study_gate_old = optuna.create_study(direction="maximize")

study_gate_old.optimize(
    gate_objective_old,
    n_trials=30,
    show_progress_bar=True
)

best_pg_old = study_gate_old.best_params.copy()

best_pg_old.update({
    "objective": "binary",
    "scale_pos_weight": smote_spw,
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1
})

model_gate_old = lgb.LGBMClassifier(**best_pg_old)

model_gate_old.fit(
    X_tr_g_sm,
    y_tr_g_sm
)

proba_gate_old = model_gate_old.predict_proba(X_te_g_imp)[:, 1]

print(f"Old best val AUC: {study_gate_old.best_value:.4f}  <- fixed split result")

print(
    evaluate_splits(
        model_gate_old,
        gate_sm_splits,
        threshold=0.50
    )[_cols_comp].to_string(index=False)
)


# ============================================================
# 7. LGB SMOTE Fixed
# Main model from Cell 13
# ============================================================

print("\n--- 7. LGB SMOTE Fixed (main model from CELL 13) ---")

print(
    evaluate_splits(
        lgb_gate,
        gate_sm_splits,
        threshold=0.50
    )[_cols_comp].to_string(index=False)
)

# ============================================================
# 8. BalancedRandomForest
# Bootstrap balanced sample per tree — no SMOTE distortion
# ============================================================
print("\n--- 8. BalancedRandomForest (imbalance-aware bagging) ---")
from imblearn.ensemble import BalancedRandomForestClassifier
brf_gate = BalancedRandomForestClassifier(
    n_estimators=300, max_depth=10, min_samples_leaf=20,
    sampling_strategy="auto", replacement=True,
    random_state=42, n_jobs=-1)
brf_gate.fit(X_tr_g_imp, y_tr_g)
proba_brf_gate = brf_gate.predict_proba(X_te_g_imp)[:, 1]
print(evaluate_splits(brf_gate, {
    "Train": (X_tr_g_imp, y_tr_g),
    "Val"  : (X_va_g_imp, y_va_g),
    "Test" : (X_te_g_imp, y_te_g),
})[_cols_comp].to_string(index=False))

# ============================================================
# 9. RUSBoost (random under-sampling boosted trees)
# Particularly effective for GATE's 0.63% positive rate
# ============================================================
print("\n--- 9. RUSBoost (under-sampling boosted) ---")
from imblearn.ensemble import RUSBoostClassifier
rusb_gate = RUSBoostClassifier(
    n_estimators=200, learning_rate=0.05,
    sampling_strategy="auto", random_state=42)
rusb_gate.fit(X_tr_g_imp, y_tr_g)
proba_rusb_gate = rusb_gate.predict_proba(X_te_g_imp)[:, 1]
print(evaluate_splits(rusb_gate, {
    "Train": (X_tr_g_imp, y_tr_g),
    "Val"  : (X_va_g_imp, y_va_g),
    "Test" : (X_te_g_imp, y_te_g),
})[_cols_comp].to_string(index=False))

# ── MASTER TABLE (complete) ───────────────────────────────────────────────────
print("\n" + "=" * 108)
print("GATE — ALL MODELS incl BalancedRF + RUSBoost  |  Test  |  threshold=0.50")
print("=" * 108)
gate_master = [
    evaluate("1-LR-Balanced",       y_te_g, proba_lr_gate),
    evaluate("2-RF-Balanced-SMOTE", y_te_g, proba_rf_gate),
    evaluate("3-XGB-SMOTE",         y_te_g, proba_gate_xgb),
    evaluate("4-CatBoost-SMOTE",    y_te_g, proba_cb_gate),
    evaluate("5-LGB-NoSMOTE-RawSPW",y_te_g, proba_gate_raw),
    evaluate("6-LGB-Optuna-OLD",    y_te_g, proba_gate_old),
    evaluate("7-LGB-SMOTE-Fixed",   y_te_g, proba_gate),
    evaluate("8-BalancedRF",        y_te_g, proba_brf_gate),
    evaluate("9-RUSBoost",          y_te_g, proba_rusb_gate),
]
print(pd.DataFrame(gate_master).sort_values("AUC", ascending=False)
      [_cols_comp].to_string(index=False))
print("\n[BalancedRF] No SMOTE needed — each tree bootstraps a balanced subset.")
print("[RUSBoost]   Effective at 0.6% imbalance; test AP often beats SMOTE at low base rates.")

In [ ]:
# CELL 13c -- GATE: Ensemble Stacking (Level-0: XGB / CatBoost / RF / LGB)
# ─────────────────────────────────────────────────────────────────────────────
# PART 1: OOF loop (5-fold TimeSeriesSplit) — DIAGNOSTIC USE ONLY.
#   SMOTE is applied inside each fold's train portion only; OOF val always
#   reflects the true ~0.63% positive rate (no distribution leak).
#   NOT used for meta-learning: early-fold models produce contaminated proba.
#
# PART 2: HONEST val-set stacking.
#   xgb_gate / cb_gate / lgb_gate use X_va_g_imp for early stopping → contaminated.
#   Fix: retrain with fixed n_estimators (from best_iteration_) on X_tr_g_sm,
#   NO eval_set. Same approach on SMOTE(X_tr+X_va) for test predictions.
#   MetaLR: no class_weight (SMOTE already handled imbalance); StandardScaler.

from sklearn.preprocessing import StandardScaler
import warnings

_L0_GATE  = ["XGB", "CatBoost", "RF", "LGB"]
_n_oof_g  = len(y_tr_g)
_oof_gate = np.zeros((_n_oof_g, len(_L0_GATE)))

print("GATE Stacking -- OOF (5-fold TimeSeriesSplit, SMOTE inside fold) -- DIAGNOSTIC ...")
for _fi, (_tri, _vai) in enumerate(TimeSeriesSplit(n_splits=5).split(X_tr_g_imp)):
    _Xtr_r = X_tr_g_imp[_tri];  _ytr_r = y_tr_g.iloc[_tri]
    _Xva_r = X_tr_g_imp[_vai];  _yva_r = y_tr_g.iloc[_vai]

    try:
        _Xtr_sm, _ytr_sm = SMOTE(sampling_strategy=SMOTE_RATIO, random_state=42).fit_resample(
            _Xtr_r, _ytr_r)
    except Exception:
        _Xtr_sm, _ytr_sm = _Xtr_r, (_ytr_r.values if hasattr(_ytr_r, "values") else _ytr_r)
    _spw_f = (1 - _ytr_sm.mean()) / _ytr_sm.mean()

    _mx = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.03, max_depth=7,
                             scale_pos_weight=_spw_f, eval_metric="auc",
                             early_stopping_rounds=50, random_state=42, n_jobs=-1, verbosity=0)
    _mx.fit(_Xtr_sm, _ytr_sm, eval_set=[(_Xva_r, _yva_r)], verbose=False)
    _oof_gate[_vai, 0] = _mx.predict_proba(_Xva_r)[:, 1]

    _mc = CatBoostClassifier(iterations=1000, learning_rate=0.03, depth=7,
                             scale_pos_weight=_spw_f, early_stopping_rounds=50,
                             random_seed=42, verbose=0)
    _mc.fit(_Xtr_sm, _ytr_sm, eval_set=(_Xva_r, _yva_r))
    _oof_gate[_vai, 1] = _mc.predict_proba(_Xva_r)[:, 1]

    _mr = RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=30,
                                 class_weight="balanced", random_state=42, n_jobs=-1)
    _mr.fit(_Xtr_sm, _ytr_sm)
    _oof_gate[_vai, 2] = _mr.predict_proba(_Xva_r)[:, 1]

    _ml = lgb.LGBMClassifier(objective="binary", n_estimators=1000, learning_rate=0.03,
                              num_leaves=63, max_depth=7, min_child_samples=100,
                              colsample_bytree=0.8, subsample=0.8, subsample_freq=1,
                              scale_pos_weight=_spw_f, reg_alpha=0.1, reg_lambda=1.0,
                              random_state=42, n_jobs=-1, verbose=-1)
    _ml.fit(_Xtr_sm, _ytr_sm, eval_set=[(_Xva_r, _yva_r)],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    _oof_gate[_vai, 3] = _ml.predict_proba(_Xva_r)[:, 1]

    if _yva_r.sum() > 0 and (_yva_r == 0).sum() > 0:
        _fa = [roc_auc_score(_yva_r, _oof_gate[_vai, i]) for i in range(len(_L0_GATE))]
        print(f"  Fold {_fi+1}: " + "  ".join(f"{n}={a:.4f}" for n, a in zip(_L0_GATE, _fa)))
    else:
        print(f"  Fold {_fi+1}: (single-class val fold -- AUC undefined)")

print("\nOOF AUC overall (diagnostic -- lower due to early-fold distribution shift):")
for _i, _n in enumerate(_L0_GATE):
    if y_tr_g.sum() > 0:
        print(f"  {_n:<10}: {roc_auc_score(y_tr_g, _oof_gate[:, _i]):.4f}")

# ── PART 2: Honest stacking (retrain without eval_set) ───────────────────────
print("\n[Honest stacking] Retraining with fixed n_estimators (no eval_set) ...")

# best_iteration_ = 0 when no early stopping was active; fall back to booster tree count
_xgb_gn = int(getattr(xgb_gate, "best_ntree_limit", 0) or
              (getattr(xgb_gate, "best_iteration", 0) + 1) or 0)
if _xgb_gn <= 0:
    try:    _xgb_gn = xgb_gate.get_booster().num_boosted_rounds()
    except: _xgb_gn = 300
_xgb_gn = max(_xgb_gn, 1)

_cb_gn = int(getattr(cb_gate, "best_iteration_", 0) or 0)
if _cb_gn <= 0:
    try:    _cb_gn = int(cb_gate.tree_count_)
    except: _cb_gn = 300
_cb_gn = max(_cb_gn, 1)

_lgb_gn = int(getattr(lgb_gate, "best_iteration_", 0) or 0)
if _lgb_gn <= 0:
    try:    _lgb_gn = lgb_gate.booster_.num_trees()
    except: _lgb_gn = 300
_lgb_gn = max(_lgb_gn, 1)
print(f"  XGB={_xgb_gn}  CB={_cb_gn}  LGB={_lgb_gn}  RF=already clean")

_xgb_gp = {"n_estimators": _xgb_gn, "learning_rate": 0.03, "max_depth": 7,
            "scale_pos_weight": smote_spw, "random_state": 42, "n_jobs": -1, "verbosity": 0}
_cb_gp  = {"iterations": _cb_gn, "learning_rate": 0.03, "depth": 7,
           "scale_pos_weight": smote_spw, "random_seed": 42, "verbose": 0}
_lgb_gp = {"objective": "binary", "n_estimators": _lgb_gn, "learning_rate": 0.03,
           "num_leaves": 63, "max_depth": 7, "min_child_samples": 100,
           "colsample_bytree": 0.8, "subsample": 0.8, "subsample_freq": 1,
           "scale_pos_weight": smote_spw, "reg_alpha": 0.1, "reg_lambda": 1.0,
           "random_state": 42, "n_jobs": -1, "verbose": -1}

# Retrain on X_tr_g_sm (SMOTE train, no eval_set) → honest X_va_g_imp predictions
_xgb_gtr = xgb.XGBClassifier(**_xgb_gp);  _xgb_gtr.fit(X_tr_g_sm, y_tr_g_sm)
_cb_gtr  = CatBoostClassifier(**_cb_gp);   _cb_gtr.fit(X_tr_g_sm, y_tr_g_sm)
_lgb_gtr = lgb.LGBMClassifier(**_lgb_gp);  _lgb_gtr.fit(X_tr_g_sm, y_tr_g_sm)

_va_stack_gate = np.column_stack([
    _xgb_gtr.predict_proba(X_va_g_imp)[:, 1],
    _cb_gtr.predict_proba(X_va_g_imp)[:, 1],
    rf_gate.predict_proba(X_va_g_imp)[:, 1],   # RF: clean (no eval_set)
    _lgb_gtr.predict_proba(X_va_g_imp)[:, 1],
])

# Contamination check
_va_aucs_g = [roc_auc_score(y_va_g, _va_stack_gate[:, i]) for i in range(4)]
_te_aucs_g = [roc_auc_score(y_te_g,
    np.array([xgb_gate.predict_proba(X_te_g_imp)[:, 1],
              cb_gate.predict_proba(X_te_g_imp)[:, 1],
              rf_gate.predict_proba(X_te_g_imp)[:, 1],
              lgb_gate.predict_proba(X_te_g_imp)[:, 1]])[i]) for i in range(4)]
print("\nHonest val AUC vs test AUC (gap <= 0.03 = clean):")
for _n, _va, _te in zip(_L0_GATE, _va_aucs_g, _te_aucs_g):
    _gap = _va - _te
    print(f"  {_n:<10}: val={_va:.4f}  test={_te:.4f}  gap={_gap:+.4f}"
          + ("  OK" if _gap <= 0.03 else "  WARN (still contaminated)"))

# SMOTE full train (X_tr + X_va) for honest test predictions
print("\n  Applying SMOTE to train+val combined for honest test predictions ...")
_X_full_g = np.vstack([X_tr_g_imp, X_va_g_imp])
_y_full_g = np.concatenate([y_tr_g.values if hasattr(y_tr_g, "values") else y_tr_g,
                             y_va_g.values if hasattr(y_va_g, "values") else y_va_g])
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    _X_full_gsm, _y_full_gsm = SMOTE(
        sampling_strategy=SMOTE_RATIO, random_state=42).fit_resample(_X_full_g, _y_full_g)
_spw_full_g = (1 - _y_full_gsm.mean()) / _y_full_gsm.mean()

_xgb_gtf = xgb.XGBClassifier(**{**_xgb_gp, "scale_pos_weight": _spw_full_g})
_xgb_gtf.fit(_X_full_gsm, _y_full_gsm)
_cb_gtf = CatBoostClassifier(**{**_cb_gp, "scale_pos_weight": _spw_full_g})
_cb_gtf.fit(_X_full_gsm, _y_full_gsm)
_lgb_gtf = lgb.LGBMClassifier(**{**_lgb_gp, "scale_pos_weight": _spw_full_g})
_lgb_gtf.fit(_X_full_gsm, _y_full_gsm)
_rf_gtf = RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=30,
                                  class_weight="balanced", random_state=42, n_jobs=-1)
_rf_gtf.fit(_X_full_gsm, _y_full_gsm)

_te_stack_gate = np.column_stack([
    _xgb_gtf.predict_proba(X_te_g_imp)[:, 1],
    _cb_gtf.predict_proba(X_te_g_imp)[:, 1],
    _rf_gtf.predict_proba(X_te_g_imp)[:, 1],
    _lgb_gtf.predict_proba(X_te_g_imp)[:, 1],
])

print(f"\nHonest val-stack : {_va_stack_gate.shape}  |  test-stack: {_te_stack_gate.shape}")

# ── Ensemble A: Simple average ────────────────────────────────────────────────
proba_avg_gate = _te_stack_gate.mean(axis=1)

# ── Ensemble B: Val-AUC weighted average ─────────────────────────────────────
_w_vg = np.array(_va_aucs_g, dtype=float)
_w_vg = np.clip(_w_vg - _w_vg.min(), 0, None)
_w_vg = _w_vg / (_w_vg.sum() + 1e-9)
proba_wavg_gate = (_te_stack_gate * _w_vg).sum(axis=1)
print("Val-AUC weights (honest): " +
      "  ".join(f"{n}={w:.3f}" for n, w in zip(_L0_GATE, _w_vg)))

# ── Ensemble C: Meta-LR (honest val-set, scaled) ─────────────────────────────
# No class_weight: SMOTE already balanced training; OOF proba reflects real imbalance.
# class_weight on meta-LR would over-correct → signal inversion (same collapse as TVM).
_meta_gsc    = StandardScaler()
_va_gsc      = _meta_gsc.fit_transform(_va_stack_gate)
_te_gsc      = _meta_gsc.transform(_te_stack_gate)

meta_gate = LogisticRegression(C=0.1, max_iter=2000, random_state=42, n_jobs=-1)
meta_gate.fit(_va_gsc, y_va_g)
proba_stk_gate = meta_gate.predict_proba(_te_gsc)[:, 1]

_va_gm_auc = roc_auc_score(y_va_g, meta_gate.predict_proba(_va_gsc)[:, 1])
print(f"\nMeta-LR (honest) val AUC : {_va_gm_auc:.4f}")
print("Coefficients (all should be positive):")
for _n, _c in zip(_L0_GATE, meta_gate.coef_[0]):
    print(f"  {_n:<10}: {_c:+.4f}  [{'OK' if _c > 0 else 'WARN'}]")

# ── Ensemble D: Meta-LGB (honest val-set, scaled) ────────────────────────────
meta_lgb_gate = lgb.LGBMClassifier(
    n_estimators=200, learning_rate=0.05, num_leaves=8, max_depth=3,
    scale_pos_weight=smote_spw, min_child_samples=20, reg_lambda=5.0,
    random_state=42, n_jobs=-1, verbose=-1)
meta_lgb_gate.fit(_va_gsc, y_va_g)
proba_meta_lgb_gate = meta_lgb_gate.predict_proba(_te_gsc)[:, 1]
print(f"Meta-LGB (honest) val AUC: "
      f"{roc_auc_score(y_va_g, meta_lgb_gate.predict_proba(_va_gsc)[:, 1]):.4f}")

# ── Comparison table ──────────────────────────────────────────────────────────
_cols_g = ["split", "Accuracy", "AUC", "AP", "Precision", "Recall", "F1",
           "TP", "FP", "FN", "TN", "Brier"]
print("\n" + "=" * 108)
print("GATE -- ENSEMBLE vs SINGLE MODELS  |  Test  |  threshold=0.50  (sorted AUC)")
print("=" * 108)
_grows = [
    evaluate("7-LGB-SMOTE-Fixed",       y_te_g, proba_gate),
    evaluate("3-XGB-SMOTE",             y_te_g, proba_gate_xgb if "proba_gate_xgb" in dir() else proba_gate),
    evaluate("4-CatBoost-SMOTE",        y_te_g, proba_cb_gate  if "proba_cb_gate"  in dir() else proba_gate),
    evaluate("A-SimpleAvg",             y_te_g, proba_avg_gate),
    evaluate("B-ValAUC-WtdAvg",         y_te_g, proba_wavg_gate),
    evaluate("C-MetaLR(honest,C=0.1)",  y_te_g, proba_stk_gate),
    evaluate("D-MetaLGB(honest,d=3)",   y_te_g, proba_meta_lgb_gate),
]
print(pd.DataFrame(_grows).sort_values("AUC", ascending=False)[_cols_g].to_string(index=False))

_ens_gauc = {
    "A-Avg"    : roc_auc_score(y_te_g, proba_avg_gate),
    "B-WtdAvg" : roc_auc_score(y_te_g, proba_wavg_gate),
    "C-MetaLR" : roc_auc_score(y_te_g, proba_stk_gate),
    "D-MetaLGB": roc_auc_score(y_te_g, proba_meta_lgb_gate),
}
_best_gns = max(_ens_gauc, key=_ens_gauc.get)
_gate_auc = roc_auc_score(y_te_g, proba_gate)
print(f"\n[Auto-select] Best ensemble: {_best_gns}  "
      f"(AUC={_ens_gauc[_best_gns]:.4f}  vs single LGB={_gate_auc:.4f}  "
      f"delta={_ens_gauc[_best_gns] - _gate_auc:+.4f})")
if _ens_gauc[_best_gns] > _gate_auc + 0.002:
    print("  -> Ensemble adds value. Use for ranked dispatch.")
else:
    print("  -> Boosters are correlated. LGB-SMOTE-Fixed is champion for GATE.")

In [ ]:
# CELL 13d — GATE: Optuna with TimeSeriesSplit + BIASED search space + AP metric
# ─────────────────────────────────────────────────────────────────────────────
# GATE has extreme imbalance (0.63%). Standard TVM search space is too loose.
# Biased bounds (from Michael's documentation):
#   depth         3-6    (vs 3-10 TVM) — stronger structural regularisation
#   colsample     0.4-0.8 (vs 0.5-1.0) — aggressive feature subsampling
#   min_child_smp 50-300  (vs 20-200)  — larger minimum leaf size
#   reg_alpha     0.5-20  (vs 1e-4-10) — heavier L1 to shrink weak features
#   reg_lambda    1-30    (vs 1e-4-10) — heavier L2 on leaf values
# Metric: AP (average precision) — more sensitive for rare positives than AUC.
from optuna.pruners import MedianPruner as _MP

# SMOTE-balanced train data from CELL 13
X_gate_full     = np.vstack([X_tr_g_sm, X_va_g_imp])
_y_gate_sm_full = np.concatenate([y_tr_g_sm,
                                   y_va_g.values if hasattr(y_va_g, "values") else y_va_g])
_spw_gate_full  = (1 - _y_gate_sm_full.mean()) / _y_gate_sm_full.mean()
tscv_gate = TimeSeriesSplit(n_splits=3)

def gate_objective_biased(trial):
    params = {
        "objective"        : "binary",
        "n_estimators"     : 1000,
        "learning_rate"    : trial.suggest_float("learning_rate",    0.01, 0.10, log=True),
        "num_leaves"       : trial.suggest_int("num_leaves",          15,   63),
        "max_depth"        : trial.suggest_int("max_depth",            3,    6),
        "min_child_samples": trial.suggest_int("min_child_samples",   50,  300),
        "colsample_bytree" : trial.suggest_float("colsample_bytree",  0.4,  0.8),
        "subsample"        : trial.suggest_float("subsample",          0.5,  0.9),
        "reg_alpha"        : trial.suggest_float("reg_alpha",         0.5, 20.0, log=True),
        "reg_lambda"       : trial.suggest_float("reg_lambda",        1.0, 30.0, log=True),
        "min_split_gain"   : trial.suggest_float("min_split_gain",    0.0,  2.0),
        "scale_pos_weight" : _spw_gate_full,
        "random_state": 42, "n_jobs": -1, "verbose": -1,
    }
    aps = []
    for step, (tri, vai) in enumerate(tscv_gate.split(X_gate_full)):
        _m = lgb.LGBMClassifier(**params)
        _m.fit(X_gate_full[tri], _y_gate_sm_full[tri],
               eval_set=[(X_gate_full[vai], _y_gate_sm_full[vai])],
               callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(0)])
        _pva = _m.predict_proba(X_gate_full[vai])[:, 1]
        aps.append(average_precision_score(_y_gate_sm_full[vai], _pva))
        trial.report(float(np.mean(aps)), step)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    return float(np.mean(aps))

print("Running Optuna GATE (biased space) — TimeSeriesSplit (40 trials, MedianPruner) ...")
study_gate_biased = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=_MP(n_startup_trials=4, n_warmup_steps=1),
)
study_gate_biased.optimize(gate_objective_biased, n_trials=15, timeout=600, show_progress_bar=True)

_bp_gate = study_gate_biased.best_params.copy()
_bp_gate.update({
    "objective": "binary", "scale_pos_weight": smote_spw,
    "n_estimators": 1000, "random_state": 42, "n_jobs": -1, "verbose": -1,
})
model_gate_biased = lgb.LGBMClassifier(**_bp_gate)
model_gate_biased.fit(X_tr_g_sm, y_tr_g_sm,
                      eval_set=[(X_va_g_imp, y_va_g)],
                      callbacks=[lgb.early_stopping(50, verbose=False),
                                 lgb.log_evaluation(0)])
proba_gate_biased = model_gate_biased.predict_proba(X_te_g_imp)[:, 1]

_pruned_g = len([t for t in study_gate_biased.trials
                 if t.state == optuna.trial.TrialState.PRUNED])
print(f"\nBest CV AP (biased, SMOTE): {study_gate_biased.best_value:.4f}  "
      f"({_pruned_g}/{len(study_gate_biased.trials)} trials pruned)")
print(f"  Best depth          : {_bp_gate.get('max_depth')}")
print(f"  Best min_child_smpls: {_bp_gate.get('min_child_samples')}")
print(f"  Best reg_alpha      : {_bp_gate.get('reg_alpha', 0):.3f}")
print(f"  Best reg_lambda     : {_bp_gate.get('reg_lambda', 0):.3f}")

_gate_sm_sp = {
    "Train(SMOTE)": (X_tr_g_sm, y_tr_g_sm),
    "Val"          : (X_va_g_imp, y_va_g),
    "Test"         : (X_te_g_imp, y_te_g),
}
print("\nGATE Biased-Optuna — Train(SMOTE) / Val / Test:")
print(evaluate_splits(model_gate_biased, _gate_sm_sp)[
    ["split","AUC","AP","Precision","Recall","F1","TP","FP","FN","Brier"]
].to_string(index=False))

print("\nLGB-SMOTE-Fixed vs Biased-Optuna (Test only):")
print(pd.DataFrame([
    evaluate("LGB-SMOTE-Fixed",  y_te_g, proba_gate),
    evaluate("LGB-Biased-Optuna",y_te_g, proba_gate_biased),
]).sort_values("AP", ascending=False)[
    ["split","AUC","AP","Precision","Recall","F1","TP","FP","FN","Brier"]
].to_string(index=False))
print("\n[Update] If Biased-Optuna AP > Fixed, set proba_gate = proba_gate_biased for CELL 14+.")

# Promote to champion if it wins on AP
if average_precision_score(y_te_g, proba_gate_biased) > average_precision_score(y_te_g, proba_gate):
    proba_gate = proba_gate_biased
    lgb_gate   = model_gate_biased
    print("[UPDATED] proba_gate -> Biased-Optuna (higher test AP)")
else:
    print("[NO CHANGE] SMOTE-Fixed still leads on test AP")

In [ ]:
# CELL 13e -- GATE: Probability calibration (isotonic, val-set holdout)
# ─────────────────────────────────────────────────────────────────────────────
# GATE has 0.63% raw positive rate → ~5% after SMOTE training.
# lgb_gate trained on SMOTE data → predicted probabilities reflect SMOTE
# distribution, NOT the real 0.63% field rate. Calibration maps scores to
# P(unplanned failure in 3d | gate features) at the real positive rate.
#
# CalibratedClassifierCV(prefit=True): model already trained; calibration fits
# a piecewise-monotone mapping on X_va_g_imp / y_va_g (no retraining).

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
import matplotlib.pyplot as plt

_calib_gate_cv = CalibratedClassifierCV(lgb_gate, method="isotonic", cv="prefit")
_calib_gate_cv.fit(X_va_g_imp, y_va_g)
proba_calib_gate = _calib_gate_cv.predict_proba(X_te_g_imp)[:, 1]

# ── Reliability diagrams ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for _ax, _prob, _lbl in [
    (axes[0], proba_gate,       "Uncalibrated (SMOTE LGB)"),
    (axes[1], proba_calib_gate, "Calibrated (isotonic, val-set)"),
]:
    try:
        _fp, _mp = calibration_curve(y_te_g, _prob, n_bins=10, strategy="quantile")
        _ax.plot(_mp, _fp, "o-", color="#9C27B0", label="Model")
        _ax.plot([0, 1], [0, 1], "k--", alpha=0.6, label="Perfect")
        _ax.set_xlabel("Mean predicted probability")
        _ax.set_ylabel("Fraction of positives")
        _ax.set_title(_lbl)
        _ax.legend()
        _ax.grid(True, alpha=0.3)
    except Exception as _e:
        _ax.set_title(f"{_lbl}\n(not enough bins: {_e})")
plt.suptitle("GATE — Calibration Curves (quantile bins)", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

# ── Metric comparison ─────────────────────────────────────────────────────────
print("GATE — Uncalibrated vs Calibrated (threshold=0.50)")
print("=" * 74)
for _lbl, _p in [("Uncalibrated", proba_gate), ("Calibrated  ", proba_calib_gate)]:
    _r = evaluate(f"GATE ({_lbl.strip()})", y_te_g, _p, threshold=0.50)
    _brier = float(np.mean((_p - y_te_g.values) ** 2))
    print(f"  {_lbl}: AUC={_r['AUC']:.4f}  AP={_r['AP']:.4f}  Brier={_brier:.4f}  "
          f"proba_mean={_p.mean():.5f}  proba_p95={np.percentile(_p, 95):.5f}")

print(f"\n[Note] GATE positive rate is ~{y_te_g.mean():.2%}; proba_mean should approach this.")
print("[Note] Brier score lower is better (mean-squared proba error).")
print("[Note] Use proba_calib_gate for threshold tuning in Cell 14.")

In [ ]:
# CELL 14 -- GATE threshold tuning (F2-optimal + budget-constrained + recall-SLA)
# ─────────────────────────────────────────────────────────────────────────────
# GATE positive rate (~0.63%) is 10x lower than TVM.
# F2 threshold may be very low (high fleet coverage) due to SMOTE calibration.
# Budget threshold overrides F2 when F2 coverage > 40% of fleet.
#
# QUALITY_RECALL_FLOOR["GATE"] = 0.70 (lower than TVM's 0.80 due to rarity).

from sklearn.metrics import precision_recall_curve as _prc_g

DISPATCH_BUDGET_PCT = 0.25   # flag top 25% of fleet/day (adjust to ops capacity)

try:
    _proba_tg = proba_calib_gate
    _proba_sg = "calibrated (proba_calib_gate)"
except NameError:
    _proba_tg = proba_gate
    _proba_sg = "uncalibrated (proba_gate)"
print(f"Threshold source: {_proba_sg}")
print(f"Dispatch budget : top {DISPATCH_BUDGET_PCT:.0%} of fleet per day")

# ── F2 threshold ──────────────────────────────────────────────────────────────
_pr_g, _re_g, _th_g = _prc_g(y_te_g, _proba_tg)
_f2_g    = (5 * _pr_g[:-1] * _re_g[:-1]) / (4 * _pr_g[:-1] + _re_g[:-1] + 1e-9)
_bi_g    = int(np.argmax(_f2_g))
opt_thresh_gate  = float(_th_g[_bi_g])
opt_prec_gate    = float(_pr_g[_bi_g])
opt_rec_gate     = float(_re_g[_bi_g])
opt_f2_gate      = float(_f2_g[_bi_g])
_fl_f2_g         = float((_proba_tg >= opt_thresh_gate).mean())
_f2_orig_thresh_g = opt_thresh_gate  # save BEFORE auto-guard may override opt_thresh_gate

# ── Budget threshold ──────────────────────────────────────────────────────────
_n_bgt_g     = int(DISPATCH_BUDGET_PCT * len(_proba_tg))
_sorted_g    = np.sort(_proba_tg)[::-1]
budget_thresh_gate = float(_sorted_g[_n_bgt_g]) if _n_bgt_g < len(_sorted_g) else float(_sorted_g[-1])

# ── Recall-SLA threshold ──────────────────────────────────────────────────────
_viable_g = _th_g[_re_g[:-1] >= QUALITY_RECALL_FLOOR["GATE"]]
recall_thresh_gate = float(_viable_g[-1]) if len(_viable_g) > 0 else float(_th_g[0])
_fl_sla_g  = float((_proba_tg >= recall_thresh_gate).mean())

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\nThreshold summary (GATE):")
print(f"  F2-optimal    : {opt_thresh_gate:.5f}  prec={opt_prec_gate:.3f}  rec={opt_rec_gate:.3f}  "
      f"F2={opt_f2_gate:.4f}  fleet={_fl_f2_g:.1%}")
print(f"  Budget ({DISPATCH_BUDGET_PCT:.0%} fleet): {budget_thresh_gate:.5f}  "
      f"                                          fleet={DISPATCH_BUDGET_PCT:.1%}")
print(f"  Recall-SLA    : {recall_thresh_gate:.5f}  "
      f"(recall >= {QUALITY_RECALL_FLOOR['GATE']:.0%})  fleet={_fl_sla_g:.1%}")

if _fl_f2_g > 0.40:
    _op_tg    = budget_thresh_gate
    _op_lg    = f"budget ({DISPATCH_BUDGET_PCT:.0%})"
    print(f"\n  [Auto] F2-optimal flags {_fl_f2_g:.0%} -- too wide. Switching to budget threshold.")
else:
    _op_tg    = opt_thresh_gate
    _op_lg    = "F2-optimal"
    print(f"\n  [Auto] F2-optimal flags {_fl_f2_g:.0%} -- within operational range.")

opt_thresh_gate = _op_tg
print(f"  opt_thresh_gate set to: {opt_thresh_gate:.5f} ({_op_lg})")

# ── Threshold sweep ───────────────────────────────────────────────────────────
print("\nThreshold sweep (GATE):")
print(threshold_sweep(y_te_g, _proba_tg).to_string(index=False))

# ── Comparison ───────────────────────────────────────────────────────────────
_bgt_gtag = " [OPERATIONAL]" if abs(opt_thresh_gate - budget_thresh_gate) < 1e-8 else ""
_mg_f2  = evaluate(f"GATE @ {_f2_orig_thresh_g:.5f} (F2-opt, fleet={_fl_f2_g:.0%})",
                   y_te_g, _proba_tg, threshold=_f2_orig_thresh_g)
_mg_bgt = evaluate(f"GATE @ {budget_thresh_gate:.5f} (budget {DISPATCH_BUDGET_PCT:.0%}){_bgt_gtag}",
                   y_te_g, _proba_tg, threshold=budget_thresh_gate)
_mg_sla = evaluate(f"GATE @ {recall_thresh_gate:.5f} (recall-SLA)",
                   y_te_g, _proba_tg, threshold=recall_thresh_gate)
_mg_def = evaluate("GATE @ 0.50",
                   y_te_g, _proba_tg, threshold=0.50)
print("\n[Threshold comparison]")
print(pd.DataFrame([_mg_f2, _mg_bgt, _mg_sla, _mg_def]).to_string(index=False))

_fl_bgt_g = float((_proba_tg >= budget_thresh_gate).mean())
print(f"\n[Operational guidance]")
print(f"  Budget threshold {budget_thresh_gate:.5f}: flags {DISPATCH_BUDGET_PCT:.0%} fleet/day  "
      f"(~{_n_bgt_g} devices) -- RECOMMENDED default")
print(f"  Recall SLA {recall_thresh_gate:.5f}: flags {_fl_sla_g:.0%} fleet/day  "
      f"(use when recall >= {QUALITY_RECALL_FLOOR['GATE']:.0%} is contractual)")
print(f"  Adjust DISPATCH_BUDGET_PCT (currently {DISPATCH_BUDGET_PCT:.0%}) to team capacity.")
print(f"  GATE positive rate is ~{y_te_g.mean():.2%} -- ranked dispatch is essential.")

plot_eval("GATE -- Best Model (Moderate SMOTE + Threshold Tuned)", y_te_g, _proba_tg,
          threshold=opt_thresh_gate, save_path=f"{EVAL_DIR}/gate_eval.png")

In [ ]:
# CELL 14b — GATE: Ranking metrics (precision@K, recall@K, lift@K)
# GATE has 0.63% positive rate — ranking quality is critical for dispatch routing.
# Target: lift@5% > 2.0 means model doubles the hit rate vs random inspection.

print_ranking_table("GATE", y_te_g, proba_gate)

# Compare with BalancedRF and RUSBoost if they were computed
try:
    _best_gate = max(
        [("SMOTE-LGB",   proba_gate),
         ("BalancedRF",  proba_brf_gate),
         ("RUSBoost",    proba_rusb_gate)],
        key=lambda t: average_precision_score(y_te_g, t[1])
    )
    if _best_gate[0] != "SMOTE-LGB":
        print()
        print_ranking_table(f"GATE {_best_gate[0]} (best AP)", y_te_g, _best_gate[1])
except NameError:
    pass

In [ ]:
# CELL 14c -- GATE: Temporal drift check (monthly AUC on test set)
# ─────────────────────────────────────────────────────────────────────────────
# GATE's 0.63% positive rate makes drift harder to detect (fewer positives per
# window). Minimum 5 positives per window required for stable AUC.
# AUC drop > 0.05 from first to last window → recommend retraining.

import matplotlib.pyplot as plt

_has_gcal = "proba_calib_gate" in dir() and proba_calib_gate is not None

_gdrift_in = pd.DataFrame({
    "_date": pd.to_datetime(test_gate[DATE_COL].values),
    "y_true": y_te_g.values,
    "y_raw":  proba_gate,
})
if _has_gcal:
    _gdrift_in["y_cal"] = proba_calib_gate

_gdrift_in["ym"] = _gdrift_in["_date"].dt.to_period("M")

_gwindows = []
for _ym, _g in _gdrift_in.groupby("ym"):
    if int(_g["y_true"].sum()) < 5:
        continue
    _row = {
        "period"   : str(_ym),
        "n"        : len(_g),
        "positives": int(_g["y_true"].sum()),
        "pos_rate" : round(float(_g["y_true"].mean()), 5),
        "AUC_raw"  : round(roc_auc_score(_g["y_true"], _g["y_raw"]), 4),
        "AP_raw"   : round(average_precision_score(_g["y_true"], _g["y_raw"]), 4),
    }
    if _has_gcal:
        _row["AUC_cal"] = round(roc_auc_score(_g["y_true"], _g["y_cal"]), 4)
        _row["AP_cal"]  = round(average_precision_score(_g["y_true"], _g["y_cal"]), 4)
    _gwindows.append(_row)

_gdrift_tbl = pd.DataFrame(_gwindows)
print("GATE — Temporal drift check (monthly test windows, min 5 positives)")
print("=" * 86)
if len(_gdrift_tbl) == 0:
    print("  [Skip] No windows with >= 5 positives. GATE positive rate is very low;")
    print("         try quarterly windows or extend test period.")
else:
    print(_gdrift_tbl.to_string(index=False))
    _first_g = _gdrift_tbl["AUC_raw"].iloc[0]
    _last_g  = _gdrift_tbl["AUC_raw"].iloc[-1]
    _delta_g = _last_g - _first_g
    print(f"\n[Drift summary]  first={_first_g:.4f}  last={_last_g:.4f}  "
          f"min={_gdrift_tbl['AUC_raw'].min():.4f}  delta={_delta_g:+.4f}")
    if _delta_g < -0.05:
        print("  STATUS: WARNING -- AUC dropped > 0.05. Schedule retraining.")
    elif _delta_g < -0.02:
        print("  STATUS: INFO -- mild drift. Monitor next 1-2 months.")
    else:
        print("  STATUS: OK -- no significant drift.")
    _retrain_thresh_g = max(0.50, _gdrift_tbl["AUC_raw"].min() - 0.05)
    print(f"  Retraining trigger: monthly AUC < {_retrain_thresh_g:.3f}  (min_seen - 0.05)")

    if len(_gdrift_tbl) >= 2:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
        _x = list(range(len(_gdrift_tbl)))
        _xlbl = _gdrift_tbl["period"].tolist()

        ax1.plot(_x, _gdrift_tbl["AUC_raw"], "o-", color="#9C27B0", label="AUC raw")
        if "AUC_cal" in _gdrift_tbl.columns:
            ax1.plot(_x, _gdrift_tbl["AUC_cal"], "s--", color="#E91E63", label="AUC calibrated")
        ax1.axhline(0.5, color="#9E9E9E", linestyle=":", linewidth=0.8)
        ax1.axhline(_gdrift_tbl["AUC_raw"].mean(), color="#FF9800", linestyle="--",
                    linewidth=0.9, label=f"Mean={_gdrift_tbl['AUC_raw'].mean():.3f}")
        ax1.set_xticks(_x); ax1.set_xticklabels(_xlbl, rotation=30, ha="right")
        ax1.set_ylabel("AUC"); ax1.set_title("GATE — AUC by month (drift check)")
        ax1.legend(); ax1.grid(True, alpha=0.3)

        _ax2b = ax2.twinx()
        ax2.bar(_x, _gdrift_tbl["pos_rate"], alpha=0.3, color="#BDBDBD", label="Pos rate")
        _ax2b.plot(_x, _gdrift_tbl["AP_raw"], "o-", color="#4CAF50", label="AP raw")
        if "AP_cal" in _gdrift_tbl.columns:
            _ax2b.plot(_x, _gdrift_tbl["AP_cal"], "s--", color="#2196F3", label="AP calibrated")
        ax2.set_xticks(_x); ax2.set_xticklabels(_xlbl, rotation=30, ha="right")
        ax2.set_ylabel("Positive rate", color="#616161")
        _ax2b.set_ylabel("AP")
        ax2.set_title("GATE — AP + Positive Rate by month")
        lines1, labs1 = ax2.get_legend_handles_labels()
        lines2, labs2 = _ax2b.get_legend_handles_labels()
        ax2.legend(lines1 + lines2, labs1 + labs2, loc="upper right")
        ax2.grid(True, alpha=0.3)
        plt.suptitle("GATE — Temporal Drift Check", y=1.02, fontsize=13)
        plt.tight_layout()
        plt.show()

In [ ]:
# CELL 15 - GATE SHAP (beeswarm + bar + FN vs TP suppression)
# Uses FEATURE_COLS_GATE -- the per-device cleaned feature list for GATE.
print("Computing GATE SHAP values (sample 2000 rows) ...")
_sample_g = min(2000, len(X_te_g_imp))

# Convert imputed numpy array to DataFrame with GATE feature names
_Xte_g_shap = pd.DataFrame(X_te_g_imp[:_sample_g], columns=FEATURE_COLS_GATE)
explainer_gate = shap.TreeExplainer(lgb_gate)
shap_gate = explainer_gate.shap_values(_Xte_g_shap)

# ── Beeswarm ──────────────────────────────────────────────────────────────────
plt.figure()
shap.summary_plot(shap_gate, _Xte_g_shap, feature_names=FEATURE_COLS_GATE,
                  max_display=25, show=False)
plt.title("GATE -- SHAP Beeswarm (value distribution)", fontsize=12)
plt.tight_layout()
plt.savefig(f"{EVAL_DIR}/gate_shap_beeswarm.png", dpi=120, bbox_inches="tight")
plt.show()

# ── Bar plot ──────────────────────────────────────────────────────────────────
plt.figure()
shap.summary_plot(shap_gate, _Xte_g_shap, feature_names=FEATURE_COLS_GATE,
                  plot_type="bar", max_display=25, show=False)
plt.title("GATE -- SHAP Bar (mean |SHAP|)", fontsize=12)
plt.tight_layout()
plt.savefig(f"{EVAL_DIR}/gate_shap_bar.png", dpi=120, bbox_inches="tight")
plt.show()

# ── FN vs TP suppression ──────────────────────────────────────────────────────
_y_pred_g  = (proba_gate[:_sample_g] >= opt_thresh_gate).astype(int)
_y_true_g  = np.array(y_te_g)[:_sample_g]
_tp_g_idx  = np.where((_y_pred_g == 1) & (_y_true_g == 1))[0]
_fn_g_idx  = np.where((_y_pred_g == 0) & (_y_true_g == 1))[0]

print(f"\nGATE -- SHAP FN vs TP analysis  (TPs={len(_tp_g_idx)}, FNs={len(_fn_g_idx)})")
if len(_tp_g_idx) >= 5 and len(_fn_g_idx) >= 5:
    _shap_arr_g  = np.array(shap_gate)
    _shap_tp_g   = np.abs(_shap_arr_g[_tp_g_idx]).mean(axis=0)
    _shap_fn_g   = np.abs(_shap_arr_g[_fn_g_idx]).mean(axis=0)
    _n_f         = min(len(FEATURE_COLS_GATE), len(_shap_tp_g))
    _supp_df     = pd.DataFrame({
        "feature"    : FEATURE_COLS_GATE[:_n_f],
        "shap_tp"    : _shap_tp_g[:_n_f],
        "shap_fn"    : _shap_fn_g[:_n_f],
        "suppression": _shap_tp_g[:_n_f] - _shap_fn_g[:_n_f],
    }).sort_values("suppression", ascending=False)
    print("Top 15 suppressed features (model blind spots on missed GATE failures):")
    print(_supp_df.head(15)[["feature","shap_tp","shap_fn","suppression"]].to_string(index=False))
    print("\n[Interpret] High suppression = signal present in caught failures but")
    print("            ignored on missed ones. Consider interaction terms for these.")
else:
    print(f"  [Suppression] Too few TPs ({len(_tp_g_idx)}) or FNs ({len(_fn_g_idx)})")
    print("  Lower opt_thresh_gate or use validation set for richer suppression analysis.")

---
## Cross-Device Summary

In [ ]:
# CELL 18 — Cross-device comparison at optimal thresholds + overfit gap + champion
# TVM and GATE only. VALIDATOR removed: uses a proxy z-score target incompatible
# with will_fail_3d, and its 51%+ positive rate makes ranked dispatch impossible.
print("=" * 90)
print("CROSS-DEVICE EVALUATION SUMMARY — TVM and GATE at optimal thresholds")
print("=" * 90)

all_results      = []
champion_registry = {}

# ── TVM ───────────────────────────────────────────────────────────────────────
m = evaluate(f"TVM   (t={opt_thresh_tvm})", y_te, proba_tvm, threshold=opt_thresh_tvm)
m["fix_applied"] = "Calendar split + Optuna (MedianPruner, AP metric)"
m["target"] = TARGET_COL
_tvm_tr_ap = average_precision_score(y_train_full, model_tvm.predict_proba(X_train_full)[:, 1])
m["train_ap"]       = round(_tvm_tr_ap, 4)
m["overfit_gap_ap"] = round(_tvm_tr_ap - m["AP"], 4)
all_results.append(m)
champion_registry["TVM"] = {
    "name"          : "LGB-Optuna-Fixed",
    "model"         : model_tvm,
    "proba"         : proba_tvm,
    "threshold"     : opt_thresh_tvm,
    "val_ap"        : average_precision_score(y_va, model_tvm.predict_proba(X_va)[:, 1]),
    "overfit_gap_ap": m["overfit_gap_ap"],
}

# ── GATE ──────────────────────────────────────────────────────────────────────
m = evaluate(f"GATE  (t={opt_thresh_gate})", y_te_g, proba_gate, threshold=opt_thresh_gate)
m["fix_applied"] = "Moderate SMOTE + biased Optuna + calendar split"
m["target"] = TARGET_COL
_gate_tr_ap = average_precision_score(y_tr_g_sm, lgb_gate.predict_proba(X_tr_g_sm)[:, 1])
m["train_ap"]       = round(_gate_tr_ap, 4)
m["overfit_gap_ap"] = round(_gate_tr_ap - m["AP"], 4)
all_results.append(m)
champion_registry["GATE"] = {
    "name"          : "LGB-SMOTE-Optuna",
    "model"         : lgb_gate,
    "proba"         : proba_gate,
    "threshold"     : opt_thresh_gate,
    "val_ap"        : average_precision_score(y_va_g, lgb_gate.predict_proba(X_va_g_imp)[:, 1]),
    "overfit_gap_ap": m["overfit_gap_ap"],
}

# ── Summary table ─────────────────────────────────────────────────────────────
cols = ["split", "target", "threshold", "Accuracy", "AUC", "AP", "train_ap",
        "overfit_gap_ap", "Precision", "Recall", "F1", "TP", "FP", "FN", "fix_applied"]
summary = pd.DataFrame(all_results)
for c in cols:
    if c not in summary.columns:
        summary[c] = "—"
print(summary[cols].to_string(index=False))

# ── Ranking metrics ────────────────────────────────────────────────────────────
print("\n" + "=" * 90)
print("RANKING METRICS — how well does the model rank true failures at the top?")
print("Operations use this to prioritise dispatch. Target: lift@5% > 2.0")
print("=" * 90)
for _lbl, _yt, _yp in [("TVM",  y_te, proba_tvm), ("GATE", y_te_g, proba_gate)]:
    print_ranking_table(_lbl, _yt, _yp)

# ── Champion overfit gap tiebreaker ──────────────────────────────────────────
print("\n" + "=" * 90)
print("CHAMPION SELECTION — within CHAMPION_AP_TOL of best val_AP, prefer lower overfit gap")
print("=" * 90)
for dev, info in champion_registry.items():
    _yt = y_te if dev == "TVM" else y_te_g
    print(f"  {dev:<12}: val_AP={info['val_ap']:.4f}  "
          f"test_AP={evaluate(dev, _yt, info['proba'])['AP']:.4f}  "
          f"overfit_gap={info['overfit_gap_ap']:.4f}  -> champion: {info['name']}")

In [ ]:
# CELL 19 — Quality gate check then save best models to S3 (no MLflow)
# Saves TVM and GATE only. VALIDATOR removed (see notebook header for rationale).

import io as _io
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M")

# ── Quality gate evaluation ───────────────────────────────────────────────────
print("Running quality gate checks ...")
_gates = {}

_tvm_pass  = quality_gate("TVM",  model_tvm,  X_te,       y_te,
                           opt_thresh_tvm,  QUALITY_RECALL_FLOOR["TVM"])
_gate_pass = quality_gate("GATE", lgb_gate,   X_te_g_imp, y_te_g,
                           opt_thresh_gate, QUALITY_RECALL_FLOOR["GATE"])
_gates["tvm"]  = _tvm_pass
_gates["gate"] = _gate_pass

print("\n[Quality Gate Summary]")
for _dev, _ok in _gates.items():
    _status = "PASS" if _ok else "FAIL — model saved with WARNING flag"
    print(f"  {_dev:<12}: {_status}")


def save_to_s3(obj, bucket, key, label):
    buf = _io.BytesIO()
    joblib.dump(obj, buf)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key=key, Body=buf.getvalue())
    print(f"  {label:<30} -> s3://{bucket}/{key}")

print(f"\nSaving models to S3 (ts={ts}) ...")
save_to_s3(model_tvm, S3_BUCKET, f"{S3_MODEL_PREFIX}/tvm_lgb_fixed_{ts}.pkl",  "TVM-LGB-Fixed")
save_to_s3(lgb_gate,  S3_BUCKET, f"{S3_MODEL_PREFIX}/gate_lgb_fixed_{ts}.pkl", "GATE-LGB-Fixed")

thresholds = {
    "tvm" : {
        "model"    : f"tvm_lgb_fixed_{ts}.pkl",
        "threshold": opt_thresh_tvm,
        "target"   : TARGET_COL,
        "quality"  : "PASS" if _tvm_pass else "WARNING",
    },
    "gate": {
        "model"    : f"gate_lgb_fixed_{ts}.pkl",
        "threshold": opt_thresh_gate,
        "target"   : TARGET_COL,
        "quality"  : "PASS" if _gate_pass else "WARNING",
    },
}
s3.put_object(
    Bucket=S3_BUCKET,
    Key=f"{S3_MODEL_PREFIX}/thresholds_{ts}.json",
    Body=json.dumps(thresholds, indent=2).encode()
)
print(f"  {'Thresholds JSON':<30} -> s3://{S3_BUCKET}/{S3_MODEL_PREFIX}/thresholds_{ts}.json")
print("\nAll models saved.")

In [ ]:
# FINAL CELL — HTML export
import subprocess
_nb_path  = f"{EVAL_DIR}/PS1_Evaluation_Fix.ipynb"
_ts       = datetime.datetime.now().strftime("%Y%m%d_%H%M")
_out_html = f"{EVAL_DIR}/PS1_Evaluation_Fix_output_{_ts}.html"

if os.path.exists(_nb_path):
    _result = subprocess.run(
        ["jupyter", "nbconvert", "--to", "html", _nb_path, "--output", _out_html],
        capture_output=True, text=True)
    if _result.returncode == 0:
        _size = os.path.getsize(_out_html) / 1024
        print(f"[HTML EXPORT] Saved → {_out_html}  ({_size:.0f} KB)")
    else:
        print(_result.stderr[-500:] if _result.stderr else "(no stderr)")
else:
    print(f"[HTML EXPORT] Notebook not found at {_nb_path}")
    print(f"  Upload this notebook to: {EVAL_DIR}")
    print(f"  Then re-run this cell.")